# Credit Card Behavior Score

## Resumo executivo

Este notebook fecha o desenvolvimento e executa a avaliação final OOT de 2019-11 a 2020-01. O protocolo foi definido antes da primeira predição OOT nesta execução, e o OOT permanece excluído de fit, tuning, seleção de features, escolha de modelo e calibração.

O CatBoost final refitado alcança ROC-AUC **0,8227**, KS **0,5027** e lift **3,50** no primeiro decil OOT. A classificação técnica é **B — boa capacidade de ordenação fora do tempo, com variabilidade temporal material**. A diferença de **0,0194** no ROC-AUC compara a referência de desenvolvimento (**0,8421**, candidato treinado no Treino e avaliado na Validação) com a avaliação OOT do modelo final refitado (**0,8227**, refit em Treino + Validação); não compara o mesmo ajuste em duas amostras. Dezembro de 2019 é a safra mais fraca, mas a ordenação agregada e a liderança sobre o benchmark permanecem.

A avaliação preserva o conjunto de 13 features, `var12_estado`, 611 árvores lidas programaticamente do candidato e ausência de recalibração. Nenhuma decisão foi alterada após observar o OOT. A transformação matemática do score 0–1000 foi implementada depois da avaliação, mas Base Score, PDO e Base Odds permanecem pendentes de decisão humana; nenhum score individual foi calculado.

## 1. Contexto e objetivo

O objetivo é construir um Behavior Score para uma carteira existente de cartão de crédito. O problema é uma classificação binária em que `Ever30Mob6 = 1` representa somente a ocorrência do evento adverso observado; seu significado de negócio não é inferido.

A saída deve ordenar clientes por risco futuro e ser convertida para uma escala de 0 a 1000, na qual maior score significa menor risco. Nesta execução, o desenvolvimento fecha um candidato único, o protocolo é definido antes da primeira predição OOT e a avaliação final fora do tempo é executada sem reabrir qualquer decisão. Depois dessa avaliação, apenas a fórmula do score é implementada; sua instanciação permanece pendente da escolha humana de Base Score, PDO e Base Odds.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from catboost import CatBoostClassifier, Pool
from IPython.display import Markdown, display
from scipy.stats import spearmanr
from sklearn.calibration import calibration_curve
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src' / 'behavior_score').exists():
    RAIZ = RAIZ.parent
if not (RAIZ / 'src' / 'behavior_score').exists():
    raise RuntimeError('Execute a partir da raiz do projeto ou da pasta notebooks/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.behavior_score.config import (
    ALVO, COLUNA_ID, COLUNA_SAFRA, FEATURES_FINAIS_ORIGINAIS,
    FEATURES_REMOVIDAS, PASTA_DADOS_BRUTOS, PASTA_FIGURAS, PASTA_TABELAS,
    SEMENTE_ALEATORIA, VARIAVEIS_CATEGORICAS, VARIAVEIS_CATEGORICAS_FINAIS,
    VARIAVEIS_MODELO, VARIAVEIS_MODELO_FINAL, VARIAVEIS_NUMERICAS,
    VARIAVEIS_NUMERICAS_FINAIS,
)
from src.behavior_score.features import (
    CODIGOS_ESPECIAIS_VAR12, ConversorCategoricoTexto,
    ExtratorVar12Continua, IndicadoresEspeciaisVar12, criar_var12_estado,
    preparar_features_catboost,
)
from src.behavior_score.metrics import (
    calcular_lift_por_decil, calcular_metricas_classificacao,
)
from src.behavior_score.scoring import calcular_behavior_score
from src.behavior_score.stability import (
    calcular_psi_categorico, calcular_psi_numerico,
)
from src.behavior_score.visualization import (
    CORES, aplicar_eixo_percentual, aplicar_layout_executivo, salvar_grafico,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 60)
print(f'Python: {sys.version.split()[0]} | raiz: {RAIZ}')

Python: 3.11.9 | raiz: C:\GitHub\datascience\projetos\credit-card-behavior-score


## 2. Dados

**Pergunta:** a fonte carregada corresponde ao contrato mínimo necessário?

A leitura é explícita, sem caminho absoluto e sem promover nenhuma coluna a índice.

In [2]:
arquivos_excel = sorted(PASTA_DADOS_BRUTOS.glob('*.xlsx'))
if len(arquivos_excel) != 1:
    raise RuntimeError(f'Esperado exatamente um Excel em data/raw; encontrados: {len(arquivos_excel)}')
caminho_base = arquivos_excel[0]
arquivo_excel = pd.ExcelFile(caminho_base, engine='openpyxl')
if 'case' not in arquivo_excel.sheet_names:
    raise RuntimeError(f'Aba case não encontrada: {arquivo_excel.sheet_names}')
base = pd.read_excel(caminho_base, sheet_name='case', index_col=None, engine='openpyxl')
base_original = base.copy(deep=True)
base['safra'] = pd.to_datetime(base[COLUNA_SAFRA].astype('Int64').astype('string'), format='%Y%m', errors='coerce')
assert not base['safra'].isna().any()
assert base.index.equals(pd.RangeIndex(len(base))) and base.index.name is None
resumo_dados = pd.DataFrame({
    'metrica': ['registros', 'colunas', 'features', 'numericas', 'categoricas', 'safras', 'inicio', 'fim'],
    'valor': [len(base), base_original.shape[1], len(VARIAVEIS_MODELO), len(VARIAVEIS_NUMERICAS),
              len(VARIAVEIS_CATEGORICAS), base['safra'].nunique(), base['safra'].min().strftime('%Y-%m'),
              base['safra'].max().strftime('%Y-%m')],
})
assert (len(base), base_original.shape[1], len(VARIAVEIS_MODELO), base['safra'].nunique()) == (200043, 18, 15, 13)
display(resumo_dados)
display(Markdown(
    f"**Análise/Interpretação:** foram carregados **{len(base):,} registros**, **{base_original.shape[1]} colunas** "
    f"e **{len(VARIAVEIS_MODELO)} features**, cobrindo {base['safra'].min():%Y-%m} a {base['safra'].max():%Y-%m}. "
    "A coluna `index` não é esperada pelo notebook final nem utilizada como feature."
))

,metrica,valor
0,registros,200043
1,colunas,18
2,features,15
3,numericas,10
4,categoricas,5
5,safras,13
6,inicio,2019-01
7,fim,2020-01


**Análise/Interpretação:** foram carregados **200,043 registros**, **18 colunas** e **15 features**, cobrindo 2019-01 a 2020-01. A coluna `index` não é esperada pelo notebook final nem utilizada como feature.

## 3. Auditoria essencial

**Pergunta:** quais características de qualidade afetam diretamente a modelagem?

In [3]:
tabela_missing = pd.DataFrame({
    'variavel': VARIAVEIS_MODELO,
    'quantidade_missing': base[VARIAVEIS_MODELO].isna().sum().values,
    'percentual_missing': base[VARIAVEIS_MODELO].isna().mean().values,
    'cardinalidade': base[VARIAVEIS_MODELO].nunique(dropna=True).values,
}).sort_values('percentual_missing', ascending=False)
auditoria_essencial = pd.DataFrame({
    'verificacao': ['IDs únicos', 'IDs repetidos', 'linhas duplicadas', 'target binário',
                    'missing no target', 'features com missing', 'features com códigos especiais'],
    'resultado': [base[COLUNA_ID].nunique(), base[COLUNA_ID].duplicated(keep=False).sum(),
                  base_original.duplicated().sum(), set(base[ALVO].unique()) == {0, 1},
                  base[ALVO].isna().sum(), int((tabela_missing['quantidade_missing'] > 0).sum()),
                  ', '.join(v for v in VARIAVEIS_NUMERICAS if base[v].isin(CODIGOS_ESPECIAIS_VAR12).any())],
})
concentracao_categoricas = pd.DataFrame([
    {'variavel': v, 'categoria_dominante': base[v].value_counts(dropna=False).index[0],
     'participacao_dominante': base[v].value_counts(dropna=False, normalize=True).iloc[0]}
    for v in VARIAVEIS_CATEGORICAS
]).sort_values('participacao_dominante', ascending=False)
display(auditoria_essencial)
display(tabela_missing.query('quantidade_missing > 0').style.format({'percentual_missing': '{:.2%}'}))
display(concentracao_categoricas.style.format({'participacao_dominante': '{:.2%}'}))
display(Markdown(
    "**Análise/Interpretação:** não há duplicidades nem repetição de IDs, e o target é binário e completo. "
    "Os pontos materiais para preparação são o missing temporal de `cat_var13`, os códigos especiais de `var12` "
    "e a concentração de `cat_var10`. Nenhum deles determina exclusão automática."
))

,verificacao,resultado
0,IDs únicos,200043
1,IDs repetidos,0
2,linhas duplicadas,0
3,target binário,True
4,missing no target,0
5,features com missing,7
6,features com códigos especiais,var12


,variavel,quantidade_missing,percentual_missing,cardinalidade
13,cat_var13,82692,41.34%,15
2,var4,1607,0.80%,37882
10,cat_var2,1323,0.66%,5
5,var8,546,0.27%,43219
6,var9,443,0.22%,13
7,var11,4,0.00%,50787
14,cat_var15,3,0.00%,13


,variavel,categoria_dominante,participacao_dominante
2,cat_var10,0.000000,81.09%
0,cat_var2,1.000000,71.59%
1,cat_var6,12.000000,58.19%
3,cat_var13,nan,41.34%
4,cat_var15,0.000000,29.02%


**Análise/Interpretação:** não há duplicidades nem repetição de IDs, e o target é binário e completo. Os pontos materiais para preparação são o missing temporal de `cat_var13`, os códigos especiais de `var12` e a concentração de `cat_var10`. Nenhum deles determina exclusão automática.

## 4. Diagnóstico temporal

**Comentário Técnico:** esta seção registra o diagnóstico que fundamentou o split. O PSI mede mudança populacional; não é regra automática de exclusão. Após a formalização do delineamento, o OOT fica congelado para decisões de desenvolvimento.

In [4]:
tabela_safras = (base.groupby('safra', as_index=False)[ALVO]
                  .agg(registros='size', eventos='sum', taxa_evento='mean'))
fig_volume = go.Figure(go.Bar(x=tabela_safras['safra'], y=tabela_safras['registros'],
                              marker_color=CORES['principal']))
aplicar_layout_executivo(fig_volume, 'Volume por safra', titulo_eixo_x='Safra',
                         titulo_eixo_y='Registros', mostrar_legenda=False)
fig_volume.show()
salvar_grafico(fig_volume, 'final_01_volume_por_safra', PASTA_FIGURAS)
fig_evento = go.Figure(go.Scatter(x=tabela_safras['safra'], y=tabela_safras['taxa_evento'],
                                  mode='lines+markers', line=dict(color=CORES['principal'], width=3)))
aplicar_layout_executivo(fig_evento, 'Taxa do evento adverso por safra', titulo_eixo_x='Safra',
                         titulo_eixo_y='Taxa do evento', mostrar_legenda=False)
aplicar_eixo_percentual(fig_evento)
fig_evento.show()
salvar_grafico(fig_evento, 'final_02_taxa_evento_por_safra', PASTA_FIGURAS)
display(Markdown(
    f"**Análise/Interpretação:** o volume cresce de **{tabela_safras.iloc[0].registros:,}** para "
    f"**{tabela_safras.iloc[-1].registros:,}**. A taxa do evento vai de **{tabela_safras.iloc[0].taxa_evento:.2%}** "
    f"a **{tabela_safras.iloc[-1].taxa_evento:.2%}**, com mudança material em 2020-01. A causa não é inferida."
))

**Análise/Interpretação:** o volume cresce de **13,936** para **18,318**. A taxa do evento vai de **10.85%** a **17.47%**, com mudança material em 2020-01. A causa não é inferida.

In [5]:
variaveis_com_missing = [v for v in VARIAVEIS_MODELO if base[v].isna().any()]
matriz_missing = base.groupby('safra')[variaveis_com_missing].agg(lambda s: s.isna().mean()).T
fig_missing = px.imshow(matriz_missing, aspect='auto',
    color_continuous_scale=[CORES['fundo'], CORES['destaque']],
    labels={'x': 'Safra', 'y': 'Feature', 'color': 'Missing'}, zmin=0, zmax=matriz_missing.max().max())
aplicar_layout_executivo(fig_missing, 'Missing por feature e safra', titulo_eixo_x='Safra',
                         titulo_eixo_y='Feature', altura=500)
fig_missing.update_coloraxes(colorbar_tickformat='.1%')
fig_missing.show()
salvar_grafico(fig_missing, 'final_03_missing_temporal', PASTA_FIGURAS)
display(Markdown(
    f"**Análise/Interpretação:** o missing de `cat_var13` cai de "
    f"**{matriz_missing.loc['cat_var13'].iloc[0]:.2%}** para **{matriz_missing.loc['cat_var13'].iloc[-1]:.2%}**. "
    "O achado exige tratamento e monitoramento, mas não demonstra leakage ou necessidade de remoção."
))

**Análise/Interpretação:** o missing de `cat_var13` cai de **78.33%** para **23.21%**. O achado exige tratamento e monitoramento, mas não demonstra leakage ou necessidade de remoção.

In [6]:
referencia_psi = base[base['safra'].between('2019-01-01', '2019-08-01')]
registros_psi = []
for variavel in VARIAVEIS_MODELO:
    for safra, comparacao in base[base['safra'] > '2019-08-01'].groupby('safra'):
        if variavel in VARIAVEIS_NUMERICAS:
            especiais = CODIGOS_ESPECIAIS_VAR12 if variavel == 'var12' else ()
            valor = calcular_psi_numerico(referencia_psi[variavel], comparacao[variavel],
                                          especiais=especiais)
        else:
            valor = calcular_psi_categorico(referencia_psi[variavel], comparacao[variavel])
        registros_psi.append({'variavel': variavel, 'safra': safra, 'psi': valor})
tabela_psi_temporal = pd.DataFrame(registros_psi)
matriz_psi = tabela_psi_temporal.pivot(index='variavel', columns='safra', values='psi')
fig_psi = px.imshow(matriz_psi, aspect='auto',
    color_continuous_scale=[CORES['fundo'], CORES['destaque']],
    labels={'x': 'Safra comparada', 'y': 'Feature', 'color': 'PSI'})
aplicar_layout_executivo(fig_psi, 'PSI exploratório versus Treino 2019-01 a 2019-08',
                         titulo_eixo_x='Safra comparada', titulo_eixo_y='Feature', altura=650)
fig_psi.show()
salvar_grafico(fig_psi, 'final_04_psi_temporal', PASTA_FIGURAS)
display(tabela_psi_temporal.groupby('variavel')['psi'].max().sort_values(ascending=False).head(8).to_frame('psi_maximo'))
display(Markdown(
    "**Comentário Técnico:** o PSI usa a implementação única de `src/behavior_score/stability.py`, "
    "com faixas definidas exclusivamente pela referência de Treino e ausência tratada como categoria "
    "própria. Esta visão é descritiva e antecede o congelamento operacional do OOT. Nenhuma decisão de "
    "feature depende dela: a tabela de decisão usa somente o PSI de Treino versus Validação."
))

,psi_maximo
variavel,
cat_var13,0.411134
cat_var10,0.276739
cat_var6,0.214342
var3,0.072508
var7,0.064982
var1,0.040133
var12,0.037198
var14,0.035414


**Comentário Técnico:** o PSI usa a implementação única de `src/behavior_score/stability.py`, com faixas definidas exclusivamente pela referência de Treino e ausência tratada como categoria própria. Esta visão é descritiva e antecede o congelamento operacional do OOT. Nenhuma decisão de feature depende dela: a tabela de decisão usa somente o PSI de Treino versus Validação.

## 5. Delineamento das amostras

A decisão humana aprovou o cenário 8/2/3: Treino 2019-01–2019-08, Validação 2019-09–2019-10 e OOT 2019-11–2020-01. O Treino mantém volume suficiente e o OOT de três meses permite uma avaliação temporal mais exigente.

**Comentário Técnico:** o OOT está congelado. Após esta tabela de caracterização, ele não é transformado, pontuado ou usado em decisões.

In [7]:
base['amostra'] = np.select([
    base['safra'].between('2019-01-01', '2019-08-01'),
    base['safra'].between('2019-09-01', '2019-10-01'),
    base['safra'].between('2019-11-01', '2020-01-01'),
], ['Treino', 'Validação', 'OOT'], default='Fora')
ordem_amostras = ['Treino', 'Validação', 'OOT']
tabela_split = (base.groupby('amostra').agg(
    primeira_safra=('safra', 'min'), ultima_safra=('safra', 'max'), safras=('safra', 'nunique'),
    registros=(ALVO, 'size'), eventos=(ALVO, 'sum'), taxa_evento=(ALVO, 'mean'))
    .reindex(ordem_amostras).reset_index())
tabela_split['percentual_populacao'] = tabela_split['registros'] / len(base)
display(tabela_split.style.format({'primeira_safra': lambda x: x.strftime('%Y-%m'),
    'ultima_safra': lambda x: x.strftime('%Y-%m'), 'taxa_evento': '{:.2%}',
    'percentual_populacao': '{:.2%}'}))
assert tabela_split['registros'].sum() == len(base)
treino = base.loc[base['amostra'].eq('Treino')].copy()
validacao = base.loc[base['amostra'].eq('Validação')].copy()
indices_oot_congelado = set(base.index[base['amostra'].eq('OOT')])
assert set(treino.index).isdisjoint(indices_oot_congelado)
assert set(validacao.index).isdisjoint(indices_oot_congelado)
print(f'Treino: {len(treino):,} | Validação: {len(validacao):,} | OOT congelado: {len(indices_oot_congelado):,}')

,amostra,primeira_safra,ultima_safra,safras,registros,eventos,taxa_evento,percentual_populacao
0,Treino,2019-01,2019-08,8,114324,13338,11.67%,57.15%
1,Validação,2019-09,2019-10,2,32621,4093,12.55%,16.31%
2,OOT,2019-11,2020-01,3,53098,7787,14.67%,26.54%


Treino: 114,324 | Validação: 32,621 | OOT congelado: 53,098


## 6. EDA orientada à modelagem

**Pergunta:** quais relações e problemas observados no Treino se mantêm na Validação e afetam a preparação?

A partir desta seção, somente Treino e Validação são utilizados.

In [8]:
def sinal_univariado_numerico(variavel):
    ref = treino[variavel].copy()
    comp = validacao[variavel].copy()
    if variavel == 'var12':
        ref = ref.mask(ref.isin(CODIGOS_ESPECIAIS_VAR12))
        comp = comp.mask(comp.isin(CODIGOS_ESPECIAIS_VAR12))
    mediana = ref.median()
    auc = roc_auc_score(validacao[ALVO], comp.fillna(mediana))
    return 2 * max(auc, 1 - auc) - 1

def sinal_univariado_categorico(variavel):
    chave_treino = treino[variavel].astype('string').fillna('__MISSING__')
    chave_validacao = validacao[variavel].astype('string').fillna('__MISSING__')
    taxas = treino.assign(_chave=chave_treino).groupby('_chave')[ALVO].mean()
    probabilidade = chave_validacao.map(taxas).fillna(treino[ALVO].mean())
    auc = roc_auc_score(validacao[ALVO], probabilidade)
    return 2 * max(auc, 1 - auc) - 1

correlacao_spearman = treino[VARIAVEIS_NUMERICAS].corr(method='spearman')
triangulo = correlacao_spearman.abs().where(np.triu(np.ones(correlacao_spearman.shape), 1).astype(bool))
pares_correlacionados = (triangulo.stack().rename('correlacao_spearman_abs').reset_index()
                         .rename(columns={'level_0': 'variavel_1', 'level_1': 'variavel_2'})
                         .sort_values('correlacao_spearman_abs', ascending=False))
display(pares_correlacionados.head(10).style.format({'correlacao_spearman_abs': '{:.3f}'}))
display(Markdown(
    "**Análise/Interpretação:** existem pares com correlação de Spearman acima de 0,80, especialmente "
    "`var1/var3`, `var5/var9`, `var8/var9` e `var4/var8`. A regularização reduz o risco para a "
    "Logística, mas redundância e estabilidade devem ser revistas antes da decisão final."
))

,variavel_1,variavel_2,correlacao_spearman_abs
1,var1,var3,0.822
36,var5,var9,0.811
56,var8,var9,0.808
25,var4,var8,0.800
57,var8,var11,0.779
37,var5,var11,0.773
39,var5,var14,0.770
26,var4,var9,0.674
35,var5,var8,0.656
67,var9,var11,0.646


**Análise/Interpretação:** existem pares com correlação de Spearman acima de 0,80, especialmente `var1/var3`, `var5/var9`, `var8/var9` e `var4/var8`. A regularização reduz o risco para a Logística, mas redundância e estabilidade devem ser revistas antes da decisão final.

In [9]:
def calcular_vif(dados: pd.DataFrame) -> pd.DataFrame:
    """Calcula o VIF de cada coluna contra as demais na matriz padronizada.

    A imputação usa a mediana da própria amostra recebida, replicando o que a
    Regressão Logística enxerga. O VIF mede dependência **linear** e complementa,
    sem substituir, a correlação de postos de Spearman.
    """

    matriz = dados.fillna(dados.median())
    matriz = (matriz - matriz.mean()) / matriz.std(ddof=0)
    registros = []
    for coluna in matriz.columns:
        resposta = matriz[coluna].to_numpy()
        preditoras = np.column_stack([np.ones(len(matriz)), matriz.drop(columns=coluna).to_numpy()])
        coeficientes, *_ = np.linalg.lstsq(preditoras, resposta, rcond=None)
        residuo = resposta - preditoras @ coeficientes
        r2 = 1 - float(residuo.var()) / float(resposta.var())
        registros.append({'variavel': coluna, 'r2_contra_demais': r2,
                          'vif': float('inf') if r2 >= 1 else 1 / (1 - r2)})
    return pd.DataFrame(registros).sort_values('vif', ascending=False).reset_index(drop=True)

numericas_para_vif = [v for v in VARIAVEIS_NUMERICAS if v != 'var12']
tabela_vif = calcular_vif(treino[numericas_para_vif])
display(tabela_vif.style.format({'r2_contra_demais': '{:.4f}', 'vif': '{:.2f}'}).hide(axis='index'))
vif_maximo = float(tabela_vif['vif'].max())
display(Markdown(
    f"**Análise/Interpretação:** o maior VIF do bloco numérico é **{vif_maximo:.2f}**, em "
    f"`{tabela_vif.iloc[0]['variavel']}`. `var12` foi excluída do cálculo porque cerca de 93% dos seus "
    "registros são códigos especiais, o que tornaria a componente contínua não representativa. "
    "Como referência usual, VIF acima de 5 indica redundância linear relevante e acima de 10 costuma ser "
    "tratado como colinearidade severa. O VIF isolado não determina exclusão: ele indica onde a leitura "
    "dos coeficientes individuais exige cautela."
))

variavel,r2_contra_demais,vif
var5,0.8467,6.52
var9,0.7954,4.89
var8,0.7858,4.67
var11,0.7267,3.66
var3,0.6961,3.29
var1,0.6839,3.16
var14,0.5922,2.45
var4,0.5143,2.06
var7,0.1382,1.16


**Análise/Interpretação:** o maior VIF do bloco numérico é **6.52**, em `var5`. `var12` foi excluída do cálculo porque cerca de 93% dos seus registros são códigos especiais, o que tornaria a componente contínua não representativa. Como referência usual, VIF acima de 5 indica redundância linear relevante e acima de 10 costuma ser tratado como colinearidade severa. O VIF isolado não determina exclusão: ele indica onde a leitura dos coeficientes individuais exige cautela.

## 7. Análise e decisão preliminar de features

Nenhum indicador isolado determina exclusão. O sinal univariado é medido na Validação com parâmetros derivados do Treino; o PSI compara Treino e Validação.

In [10]:
def percentual_mascarado_apos_tratamento(amostra: pd.DataFrame, variavel: str, tipo: str) -> float:
    """Fração de registros cuja componente numérica precisa ser imputada.

    Para categóricas o resultado é zero: a ausência vira categoria explícita e
    nada é imputado. Para ``var12`` somam-se ausência nativa e códigos especiais,
    porque ambos são mascarados antes da imputação.
    """

    serie = amostra[variavel]
    if tipo == 'categorica':
        return 0.0
    if variavel == 'var12':
        return float((serie.isna() | serie.isin(CODIGOS_ESPECIAIS_VAR12)).mean())
    return float(serie.isna().mean())

variaveis_redundantes = set(pares_correlacionados.query('correlacao_spearman_abs >= 0.80')[['variavel_1', 'variavel_2']].to_numpy().ravel())
registros_features = []
for variavel in VARIAVEIS_MODELO:
    tipo = 'numerica' if variavel in VARIAVEIS_NUMERICAS else 'categorica'
    if tipo == 'numerica':
        sinal = sinal_univariado_numerico(variavel)
        especiais = CODIGOS_ESPECIAIS_VAR12 if variavel == 'var12' else ()
        estabilidade = calcular_psi_numerico(treino[variavel], validacao[variavel], especiais=especiais)
    else:
        sinal = sinal_univariado_categorico(variavel)
        estabilidade = calcular_psi_categorico(treino[variavel], validacao[variavel])
    if variavel == 'var12':
        decisao, observacao = 'manter_com_tratamento', 'separar componente contínua e códigos especiais'
    elif variavel == 'cat_var13':
        decisao, observacao = 'avaliar', 'missing explícito; exige teste condicional à safra'
    elif variavel == 'cat_var10':
        decisao, observacao = 'avaliar', 'concentração e drift; exige ablação'
    elif variavel == 'cat_var6':
        decisao, observacao = 'avaliar', 'PSI Treino-Validação material; exige ablação'
    elif variavel in variaveis_redundantes:
        decisao, observacao = 'avaliar', 'redundância alta; exige VIF e ablação'
    else:
        decisao, observacao = 'manter', 'sem bloqueador isolado no desenvolvimento'
    registros_features.append({
        'variavel': variavel, 'tipo': tipo, 'missing_treino': treino[variavel].isna().mean(),
        'missing_validacao': validacao[variavel].isna().mean(),
        'mascarado_apos_tratamento_treino': percentual_mascarado_apos_tratamento(treino, variavel, tipo),
        'sinal_preditivo_gini_abs': sinal,
        'estabilidade_psi': estabilidade, 'observacao': observacao, 'decisao_preliminar': decisao,
    })
tabela_features = pd.DataFrame(registros_features).sort_values(['decisao_preliminar', 'variavel'])
display(tabela_features.style.format({'missing_treino': '{:.2%}', 'missing_validacao': '{:.2%}',
    'mascarado_apos_tratamento_treino': '{:.2%}',
    'sinal_preditivo_gini_abs': '{:.3f}', 'estabilidade_psi': '{:.3f}'}))
display(tabela_features['decisao_preliminar'].value_counts().rename_axis('decisao').to_frame('features'))
display(Markdown(
    "**Comentário Técnico:** `missing_treino` mede apenas a ausência **original**. "
    "`mascarado_apos_tratamento_treino` mede quanto da componente numérica precisa ser imputada **depois** "
    "do tratamento aplicado, e é a coluna relevante para `var12`. O sinal univariado usa "
    "`2 * max(AUC, 1-AUC) - 1`: é não negativo por construção e não informa direção, portanto serve como "
    "triagem e não como prova de contribuição. Nenhuma decisão é fechada aqui. As marcas `avaliar` serão "
    "resolvidas com VIF, ablação e teste condicional à safra nas seções seguintes, e a tabela definitiva "
    "aparece na seção 17."
))

,variavel,tipo,missing_treino,missing_validacao,mascarado_apos_tratamento_treino,sinal_preditivo_gini_abs,estabilidade_psi,observacao,decisao_preliminar
12,cat_var10,categorica,0.00%,0.00%,0.00%,0.237,0.165,concentração e drift; exige ablação,avaliar
13,cat_var13,categorica,53.18%,28.92%,0.00%,0.107,0.256,missing explícito; exige teste condicional à safra,avaliar
11,cat_var6,categorica,0.00%,0.00%,0.00%,0.275,0.110,PSI Treino-Validação material; exige ablação,avaliar
0,var1,numerica,0.00%,0.00%,0.00%,0.269,0.027,redundância alta; exige VIF e ablação,avaliar
1,var3,numerica,0.00%,0.00%,0.00%,0.249,0.032,redundância alta; exige VIF e ablação,avaliar
2,var4,numerica,0.87%,0.81%,0.87%,0.290,0.002,redundância alta; exige VIF e ablação,avaliar
3,var5,numerica,0.00%,0.00%,0.00%,0.424,0.003,redundância alta; exige VIF e ablação,avaliar
5,var8,numerica,0.29%,0.30%,0.29%,0.317,0.001,redundância alta; exige VIF e ablação,avaliar
6,var9,numerica,0.23%,0.25%,0.23%,0.418,0.002,redundância alta; exige VIF e ablação,avaliar
14,cat_var15,categorica,0.00%,0.00%,0.00%,0.188,0.017,sem bloqueador isolado no desenvolvimento,manter


,features
decisao,
avaliar,9
manter,5
manter_com_tratamento,1


**Comentário Técnico:** `missing_treino` mede apenas a ausência **original**. `mascarado_apos_tratamento_treino` mede quanto da componente numérica precisa ser imputada **depois** do tratamento aplicado, e é a coluna relevante para `var12`. O sinal univariado usa `2 * max(AUC, 1-AUC) - 1`: é não negativo por construção e não informa direção, portanto serve como triagem e não como prova de contribuição. Nenhuma decisão é fechada aqui. As marcas `avaliar` serão resolvidas com VIF, ablação e teste condicional à safra nas seções seguintes, e a tabela definitiva aparece na seção 17.

### 7.1 `var12` — caracterização e tratamento

**Pergunta:** o que `var12` realmente contém e o que a transformação escolhida faz com ela?

Manter `99997`, `99998` e `99999` como números contínuos imporia à Regressão Logística distância e ordem artificiais. A estratégia adotada cria um indicador para cada código, mascara esses códigos na componente contínua e imputa a mediana aprendida no Treino. O CatBoost recebe a mesma decomposição.

**Comentário Técnico:** a tabela de caracterização abaixo corrige uma leitura enganosa. `var12` não possui ausência original, mas a quase totalidade dos seus registros é código especial. Depois do mascaramento, a maior parte da componente contínua passa a ser imputada, e a magnitude contínua só é observada de fato no grupo regular. Por isso `missing original = 0%` **não** representa a transformação efetiva.

In [11]:
def resumir_var12(amostra, nome):
    grupo = pd.Series('regular', index=amostra.index, dtype='string')
    for codigo in CODIGOS_ESPECIAIS_VAR12:
        grupo.loc[amostra['var12'].eq(codigo)] = str(codigo)
    grupo.loc[amostra['var12'].isna()] = 'ausente'
    resumo = (amostra.assign(grupo_var12=grupo).groupby('grupo_var12')[ALVO]
              .agg(registros='size', eventos='sum', taxa_evento='mean').reset_index().assign(amostra=nome))
    resumo['participacao'] = resumo['registros'] / len(amostra)
    return resumo

tabela_var12_dev = pd.concat([resumir_var12(treino, 'Treino'),
                              resumir_var12(validacao, 'Validação')], ignore_index=True)
display(tabela_var12_dev.style.format({'taxa_evento': '{:.2%}', 'participacao': '{:.2%}'}).hide(axis='index'))

registros_caracterizacao = []
for nome, amostra in [('Treino', treino), ('Validação', validacao)]:
    serie = amostra['var12']
    eh_especial = serie.isin(CODIGOS_ESPECIAIS_VAR12)
    registros_caracterizacao.append({
        'amostra': nome,
        'registros': len(serie),
        'ausencia_original': float(serie.isna().mean()),
        'codigos_especiais': float(eh_especial.mean()),
        'grupo_regular': float((~eh_especial & serie.notna()).mean()),
        'mascarado_apos_tratamento': float((eh_especial | serie.isna()).mean()),
    })
caracterizacao_var12 = pd.DataFrame(registros_caracterizacao)
display(caracterizacao_var12.style.format({
    'ausencia_original': '{:.2%}', 'codigos_especiais': '{:.2%}',
    'grupo_regular': '{:.2%}', 'mascarado_apos_tratamento': '{:.2%}'}).hide(axis='index'))

pipeline_var12_univariada = Pipeline([
    ('preparacao', ColumnTransformer([
        ('continua', Pipeline([('extrator', ExtratorVar12Continua()),
                               ('imputador', SimpleImputer(strategy='median')),
                               ('escala', StandardScaler())]), ['var12']),
        ('especiais', IndicadoresEspeciaisVar12(), ['var12']),
    ])),
    ('modelo', LogisticRegression(C=1.0, l1_ratio=0.0, max_iter=1000,
                                   random_state=SEMENTE_ALEATORIA)),
])
pipeline_var12_univariada.fit(treino[['var12']], treino[ALVO])
prob_var12_conjunto = pipeline_var12_univariada.predict_proba(validacao[['var12']])[:, 1]
metricas_var12_conjunto = calcular_metricas_classificacao(validacao[ALVO], prob_var12_conjunto)
mediana_var12_treino = float(
    pipeline_var12_univariada.named_steps['preparacao']
    .named_transformers_['continua'].named_steps['imputador'].statistics_[0]
)
diagnostico_var12 = pd.DataFrame([
    {'representacao': 'componente contínua isolada',
     'gini_validacao': sinal_univariado_numerico('var12')},
    {'representacao': 'contínua + três indicadores especiais',
     'gini_validacao': metricas_var12_conjunto['gini']},
])
display(diagnostico_var12.style.format({'gini_validacao': '{:.4f}'}).hide(axis='index'))
percentual_especial_treino = float(caracterizacao_var12.query("amostra == 'Treino'")['codigos_especiais'].iloc[0])
percentual_mascarado_treino = float(caracterizacao_var12.query("amostra == 'Treino'")['mascarado_apos_tratamento'].iloc[0])
display(Markdown(
    f"**Análise/Interpretação:** no Treino, **{percentual_especial_treino:.2%}** dos registros são códigos "
    f"especiais e apenas **{1 - percentual_especial_treino:.2%}** formam o grupo regular. Depois do "
    f"mascaramento, **{percentual_mascarado_treino:.2%}** da componente contínua passa a ser imputada com a "
    f"mediana do Treino (**{mediana_var12_treino:.4f}**), aprendida somente sobre o grupo regular. "
    f"O Gini da componente contínua isolada é **{diagnostico_var12.iloc[0]['gini_validacao']:.4f}**, "
    f"praticamente nulo, enquanto a representação completa atinge "
    f"**{diagnostico_var12.iloc[1]['gini_validacao']:.4f}**. O sinal de `var12` está nos códigos e não na "
    "magnitude contínua. As taxas do evento por grupo são muito distintas entre si, o que sustenta manter "
    "indicadores específicos. A associação não demonstra leakage nem revela o significado da variável."
))

grupo_var12,registros,eventos,taxa_evento,amostra,participacao
99997,12847,2907,22.63%,Treino,11.24%
99998,31244,2714,8.69%,Treino,27.33%
99999,63397,6005,9.47%,Treino,55.45%
regular,6836,1712,25.04%,Treino,5.98%
99997,3791,938,24.74%,Validação,11.62%
99998,10061,901,8.96%,Validação,30.84%
99999,16413,1645,10.02%,Validação,50.31%
regular,2356,609,25.85%,Validação,7.22%


amostra,registros,ausencia_original,codigos_especiais,grupo_regular,mascarado_apos_tratamento
Treino,114324,0.00%,94.02%,5.98%,94.02%
Validação,32621,0.00%,92.78%,7.22%,92.78%


representacao,gini_validacao
componente contínua isolada,0.0091
contínua + três indicadores especiais,0.2331


**Análise/Interpretação:** no Treino, **94.02%** dos registros são códigos especiais e apenas **5.98%** formam o grupo regular. Depois do mascaramento, **94.02%** da componente contínua passa a ser imputada com a mediana do Treino (**1.0000**), aprendida somente sobre o grupo regular. O Gini da componente contínua isolada é **0.0091**, praticamente nulo, enquanto a representação completa atinge **0.2331**. O sinal de `var12` está nos códigos e não na magnitude contínua. As taxas do evento por grupo são muito distintas entre si, o que sustenta manter indicadores específicos. A associação não demonstra leakage nem revela o significado da variável.

### 7.2 `cat_var13` — teste condicional à safra

**Pergunta:** a associação entre `cat_var13` ausente e o evento adverso permanece **dentro** de cada safra?

**Comentário Técnico:** a comparação agregada Treino versus Validação é insuficiente para responder isso. A proporção de ausência de `cat_var13` cai de forma acentuada ao longo do tempo e, no mesmo período, a taxa do evento sobe. Se a associação existisse apenas **entre** safras, `ausente` funcionaria como proxy da safra e não como informação do cliente. Condicionar à safra mantém o tempo constante e separa as duas hipóteses. A análise usa somente Treino e Validação.

O critério é registrado antes da execução: consistência de direção igual ou superior a 80% das safras é classificada como **sinal próprio**; abaixo disso, como **proxy temporal**.

In [12]:
registros_cat13_safra = []
for nome, amostra in [('Treino', treino), ('Validação', validacao)]:
    for safra, grupo in amostra.groupby('safra'):
        eh_ausente = grupo['cat_var13'].isna()
        quantidade_ausente = int(eh_ausente.sum())
        quantidade_observado = int((~eh_ausente).sum())
        taxa_ausente = grupo.loc[eh_ausente, ALVO].mean() if quantidade_ausente else np.nan
        taxa_observado = grupo.loc[~eh_ausente, ALVO].mean() if quantidade_observado else np.nan
        registros_cat13_safra.append({
            'amostra': nome, 'safra': safra,
            'quantidade_ausente': quantidade_ausente, 'quantidade_observado': quantidade_observado,
            'participacao_ausente': quantidade_ausente / len(grupo),
            'taxa_evento_ausente': taxa_ausente, 'taxa_evento_observado': taxa_observado,
            'diferenca_pp': (taxa_ausente - taxa_observado) * 100,
        })
tabela_cat13_safra = pd.DataFrame(registros_cat13_safra).sort_values('safra').reset_index(drop=True)
display(tabela_cat13_safra.style.format({
    'safra': lambda x: x.strftime('%Y-%m'), 'participacao_ausente': '{:.2%}',
    'taxa_evento_ausente': '{:.2%}', 'taxa_evento_observado': '{:.2%}',
    'diferenca_pp': '{:+.2f}'}).hide(axis='index'))

fig_cat13 = go.Figure()
fig_cat13.add_trace(go.Scatter(x=tabela_cat13_safra['safra'], y=tabela_cat13_safra['taxa_evento_ausente'],
    mode='lines+markers', name='cat_var13 ausente', line=dict(color=CORES['destaque'], width=3)))
fig_cat13.add_trace(go.Scatter(x=tabela_cat13_safra['safra'], y=tabela_cat13_safra['taxa_evento_observado'],
    mode='lines+markers', name='cat_var13 observado', line=dict(color=CORES['principal'], width=3)))
aplicar_layout_executivo(fig_cat13, 'Taxa do evento por safra: cat_var13 ausente versus observado',
    subtitulo='Somente Treino e Validação; o OOT permanece congelado',
    titulo_eixo_x='Safra', titulo_eixo_y='Taxa do evento')
aplicar_eixo_percentual(fig_cat13)
fig_cat13.show()
salvar_grafico(fig_cat13, 'final_10_cat_var13_por_safra', PASTA_FIGURAS)

diferencas_cat13 = tabela_cat13_safra['diferenca_pp'].dropna()
safras_ausente_menor = int((diferencas_cat13 < 0).sum())
safras_ausente_maior = int((diferencas_cat13 > 0).sum())
consistencia_cat13 = max(safras_ausente_menor, safras_ausente_maior) / len(diferencas_cat13)
direcao_cat13 = 'menor' if safras_ausente_menor >= safras_ausente_maior else 'maior'
evidencia_cat13 = 'sinal próprio' if consistencia_cat13 >= 0.8 else 'proxy temporal'
resumo_cat13 = pd.DataFrame([{
    'safras_avaliadas': len(diferencas_cat13),
    'safras_com_ausente_menor': safras_ausente_menor,
    'safras_com_ausente_maior': safras_ausente_maior,
    'consistencia_de_direcao': consistencia_cat13,
    'diferenca_media_pp': diferencas_cat13.mean(),
    'diferenca_minima_pp': diferencas_cat13.min(),
    'diferenca_maxima_pp': diferencas_cat13.max(),
    'evidencia': evidencia_cat13,
}])
display(resumo_cat13.style.format({'consistencia_de_direcao': '{:.0%}',
    'diferenca_media_pp': '{:+.2f}', 'diferenca_minima_pp': '{:+.2f}',
    'diferenca_maxima_pp': '{:+.2f}'}).hide(axis='index'))
display(Markdown(
    f"**Análise/Interpretação:** em **{max(safras_ausente_menor, safras_ausente_maior)} de "
    f"{len(diferencas_cat13)}** safras a taxa do evento no grupo ausente é **{direcao_cat13}** que no grupo "
    f"observado, o que corresponde a **{consistencia_cat13:.0%}** de consistência de direção. A diferença "
    f"média é de **{diferencas_cat13.mean():+.2f} p.p.**, variando entre "
    f"**{diferencas_cat13.min():+.2f}** e **{diferencas_cat13.max():+.2f} p.p.**. "
    f"Pelo critério registrado antes da execução, a evidência é de **{evidencia_cat13}**. "
    "Condicionar à safra mantém o tempo constante e remove a composição temporal da comparação, mas não "
    "estabelece causalidade nem revela o significado da variável."
))

amostra,safra,quantidade_ausente,quantidade_observado,participacao_ausente,taxa_evento_ausente,taxa_evento_observado,diferenca_pp
Treino,2019-01,10916,3020,78.33%,10.95%,10.50%,+0.45
Treino,2019-02,9675,4163,69.92%,11.76%,11.12%,+0.64
Treino,2019-03,8486,5297,61.57%,11.47%,11.80%,-0.33
Treino,2019-04,7639,6186,55.25%,11.51%,12.87%,-1.36
Treino,2019-05,6893,7289,48.60%,10.68%,12.70%,-2.03
Treino,2019-06,6228,8275,42.94%,11.46%,12.33%,-0.86
Treino,2019-07,5626,9182,37.99%,10.24%,12.64%,-2.41
Treino,2019-08,5329,10120,34.49%,10.32%,12.54%,-2.22
Validação,2019-09,4933,11201,30.58%,10.70%,13.08%,-2.38
Validação,2019-10,4500,11987,27.29%,10.51%,13.57%,-3.06


safras_avaliadas,safras_com_ausente_menor,safras_com_ausente_maior,consistencia_de_direcao,diferenca_media_pp,diferenca_minima_pp,diferenca_maxima_pp,evidencia
10,8,2,80%,-1.36,-3.06,+0.64,sinal próprio


**Análise/Interpretação:** em **8 de 10** safras a taxa do evento no grupo ausente é **menor** que no grupo observado, o que corresponde a **80%** de consistência de direção. A diferença média é de **-1.36 p.p.**, variando entre **-3.06** e **+0.64 p.p.**. Pelo critério registrado antes da execução, a evidência é de **sinal próprio**. Condicionar à safra mantém o tempo constante e remove a composição temporal da comparação, mas não estabelece causalidade nem revela o significado da variável.

## 8. Preparação dos dados

Todos os parâmetros são ajustados somente no Treino. Numéricas recebem mediana, indicadores de missing quando existentes e escala. Categóricas viram texto com `__MISSING__` explícito e one-hot desconhecido é ignorado. Não há balanceamento ou reamostragem.

In [13]:
X_treino = treino[VARIAVEIS_MODELO]
y_treino = treino[ALVO]
X_validacao = validacao[VARIAVEIS_MODELO]
y_validacao = validacao[ALVO]
numericas_sem_var12 = [v for v in VARIAVEIS_NUMERICAS if v != 'var12']
preprocessador_logistico = ColumnTransformer([
    ('numericas', Pipeline([('imputador', SimpleImputer(strategy='median', add_indicator=True)),
                             ('escala', StandardScaler())]), numericas_sem_var12),
    ('var12_continua', Pipeline([('extrator', ExtratorVar12Continua()),
                                  ('imputador', SimpleImputer(strategy='median')),
                                  ('escala', StandardScaler())]), ['var12']),
    ('var12_indicadores', IndicadoresEspeciaisVar12(), ['var12']),
    ('categoricas', Pipeline([('texto', ConversorCategoricoTexto()),
                               ('one_hot', OneHotEncoder(handle_unknown='ignore'))]),
     VARIAVEIS_CATEGORICAS),
])
assert set(X_treino.index).isdisjoint(indices_oot_congelado)
assert set(X_validacao.index).isdisjoint(indices_oot_congelado)
print(f'Matrizes de desenvolvimento: Treino {X_treino.shape}; Validação {X_validacao.shape}')

Matrizes de desenvolvimento: Treino (114324, 15); Validação (32621, 15)


## 9. Ressalva metodológica sobre o uso da Validação

**Comentário Técnico:** a amostra de Validação participa de cinco papéis neste desenvolvimento:

1. `eval_set` do early stopping do CatBoost;
2. seleção entre as configurações de hiperparâmetros do CatBoost e da Regressão Logística;
3. medição do sinal univariado e do PSI que alimentam a triagem de features;
4. experimentos de ablação;
5. diagnóstico de calibração.

Consequentemente, **as métricas de Validação são métricas de desenvolvimento e podem carregar algum otimismo**. O efeito é maior no CatBoost, que recebe o truncamento do número de árvores no ponto de melhor desempenho na própria Validação — um ajuste que a Regressão Logística não recebe.

A avaliação independente de generalização será feita **somente no OOT**, uma única vez, sobre a configuração congelada ao final deste notebook. O desenho Treino/Validação não é refeito nesta etapa.

## 10. Regressão Logística — benchmark

O benchmark usa distribuição natural do evento e regularização L2. O ajuste controlado compara somente `C = [0.01, 0.1, 1.0, 10.0]`; todo o pré-processamento é reajustado exclusivamente no Treino em cada configuração.

Além do desempenho, a seção verifica a **estabilidade de sinal** dos coeficientes de `var1` e `var3`, o par com maior correlação de Spearman da base. Instabilidade de sinal sob variação da regularização é a assinatura clássica de colinearidade problemática.

In [14]:
# Evidência histórica do tuning já concluído; não é recalculada neste fechamento.
tabela_tuning_logistica = pd.DataFrame([
    {'C': 0.01, 'roc_auc_treino': 0.8155108162, 'gini_treino': 0.6310216325, 'ks_treino': 0.4890898681, 'pr_auc_treino': 0.4715775829, 'brier_treino': 0.0804073943, 'roc_auc_validacao': 0.8187376941, 'gini_validacao': 0.6374753882, 'ks_validacao': 0.4912532943, 'pr_auc_validacao': 0.5337351140, 'brier_validacao': 0.0808023267, 'gap_roc_auc': -0.0032268779},
    {'C': 0.10, 'roc_auc_treino': 0.8160042587, 'gini_treino': 0.6320085175, 'ks_treino': 0.4916885501, 'pr_auc_treino': 0.4729626401, 'brier_treino': 0.0802721717, 'roc_auc_validacao': 0.8188189855, 'gini_validacao': 0.6376379710, 'ks_validacao': 0.4884469250, 'pr_auc_validacao': 0.5344337741, 'brier_validacao': 0.0807465690, 'gap_roc_auc': -0.0028147268},
    {'C': 1.00, 'roc_auc_treino': 0.8159972110, 'gini_treino': 0.6319944219, 'ks_treino': 0.4916687008, 'pr_auc_treino': 0.47304041796, 'brier_treino': 0.0802640033, 'roc_auc_validacao': 0.8187381223, 'gini_validacao': 0.6374762446, 'ks_validacao': 0.4883401294, 'pr_auc_validacao': 0.5343690024, 'brier_validacao': 0.0807556507, 'gap_roc_auc': -0.0027409113},
    {'C': 10.0, 'roc_auc_treino': 0.8159950015, 'gini_treino': 0.6319900031, 'ks_treino': 0.4915880787, 'pr_auc_treino': 0.4730573888, 'brier_treino': 0.0802631662, 'roc_auc_validacao': 0.8187175853, 'gini_validacao': 0.6374351707, 'ks_validacao': 0.4880168051, 'pr_auc_validacao': 0.5343577079, 'brier_validacao': 0.0807598170, 'gap_roc_auc': -0.0027225838},
])
c_logistica_escolhido = 0.1
pipeline_logistica = Pipeline([('preprocessamento', clone(preprocessador_logistico)),
    ('modelo', LogisticRegression(C=c_logistica_escolhido, l1_ratio=0.0, max_iter=2000,
                                   random_state=SEMENTE_ALEATORIA))])
pipeline_logistica.fit(X_treino, y_treino)
prob_log_treino = pipeline_logistica.predict_proba(X_treino)[:, 1]
prob_log_validacao = pipeline_logistica.predict_proba(X_validacao)[:, 1]
display(tabela_tuning_logistica.style.format({
    coluna: '{:.4f}' for coluna in tabela_tuning_logistica.columns if coluna != 'C'
}))
print(f'Regressão Logística escolhida: C={c_logistica_escolhido:g}.')

,C,roc_auc_treino,gini_treino,ks_treino,pr_auc_treino,brier_treino,roc_auc_validacao,gini_validacao,ks_validacao,pr_auc_validacao,brier_validacao,gap_roc_auc
0,0.010000,0.8155,0.6310,0.4891,0.4716,0.0804,0.8187,0.6375,0.4913,0.5337,0.0808,-0.0032
1,0.100000,0.8160,0.6320,0.4917,0.4730,0.0803,0.8188,0.6376,0.4884,0.5344,0.0807,-0.0028
2,1.000000,0.8160,0.6320,0.4917,0.4730,0.0803,0.8187,0.6375,0.4883,0.5344,0.0808,-0.0027
3,10.000000,0.8160,0.6320,0.4916,0.4731,0.0803,0.8187,0.6374,0.4880,0.5344,0.0808,-0.0027


Regressão Logística escolhida: C=0.1.


In [15]:
def obter_coeficiente(pipeline_ajustado, sufixo: str) -> float:
    """Recupera o coeficiente cujo nome transformado termina com ``sufixo``."""

    nomes = pipeline_ajustado.named_steps['preprocessamento'].get_feature_names_out()
    coeficientes = pipeline_ajustado.named_steps['modelo'].coef_[0]
    correspondencias = [c for n, c in zip(nomes, coeficientes) if n.endswith(sufixo)]
    if len(correspondencias) != 1:
        raise ValueError(f'Esperada exatamente uma feature terminando em {sufixo}.')
    return float(correspondencias[0])

# Coeficientes preservados do tuning anterior; nenhuma nova busca é executada.
tabela_coeficientes_c = pd.DataFrame([
    {'C': 0.01, 'coeficiente_var1': -1.1438486409, 'coeficiente_var3': 0.9234435284},
    {'C': 0.10, 'coeficiente_var1': -1.1891080442, 'coeficiente_var3': 0.9734206168},
    {'C': 1.00, 'coeficiente_var1': -1.1953441877, 'coeficiente_var3': 0.9794806721},
    {'C': 10.0, 'coeficiente_var1': -1.1973028141, 'coeficiente_var3': 0.9814816517},
])
tabela_coeficientes_c['soma_dos_coeficientes'] = (tabela_coeficientes_c['coeficiente_var1'] +
                                                   tabela_coeficientes_c['coeficiente_var3'])
tabela_coeficientes_c['razao_absoluta_var1_var3'] = (tabela_coeficientes_c['coeficiente_var1'].abs() /
                                                       tabela_coeficientes_c['coeficiente_var3'].abs())
display(tabela_coeficientes_c.style.format({'C': '{:g}', 'coeficiente_var1': '{:+.4f}',
    'coeficiente_var3': '{:+.4f}', 'soma_dos_coeficientes': '{:+.4f}',
    'razao_absoluta_var1_var3': '{:.3f}'}).hide(axis='index'))
sinais_var1 = set(np.sign(tabela_coeficientes_c['coeficiente_var1']))
sinais_var3 = set(np.sign(tabela_coeficientes_c['coeficiente_var3']))
sinal_estavel = len(sinais_var1) == 1 and len(sinais_var3) == 1
amplitude_var1 = float(tabela_coeficientes_c['coeficiente_var1'].max() - tabela_coeficientes_c['coeficiente_var1'].min())
amplitude_var3 = float(tabela_coeficientes_c['coeficiente_var3'].max() - tabela_coeficientes_c['coeficiente_var3'].min())
display(Markdown(
    f"**Análise/Interpretação:** ao variar `C` em três ordens de grandeza, o sinal de `var1` e `var3` "
    f"**{'permanece constante' if sinal_estavel else 'muda'}**. A amplitude do coeficiente é "
    f"**{amplitude_var1:.4f}** para `var1` e **{amplitude_var3:.4f}** para `var3`. "
    "Coeficientes de sinais opostos em variáveis fortemente correlacionadas indicam que o modelo utiliza "
    "principalmente o **contraste** entre as duas, e não uma cópia redundante da mesma informação. "
    "A conclusão sobre redundância é consolidada na seção 16, com VIF e ablação."
))

C,coeficiente_var1,coeficiente_var3,soma_dos_coeficientes,razao_absoluta_var1_var3
0.01,-1.1438,+0.9234,-0.2204,1.239
0.1,-1.1891,+0.9734,-0.2157,1.222
1,-1.1953,+0.9795,-0.2159,1.220
10,-1.1973,+0.9815,-0.2158,1.220


**Análise/Interpretação:** ao variar `C` em três ordens de grandeza, o sinal de `var1` e `var3` **permanece constante**. A amplitude do coeficiente é **0.0535** para `var1` e **0.0580** para `var3`. Coeficientes de sinais opostos em variáveis fortemente correlacionadas indicam que o modelo utiliza principalmente o **contraste** entre as duas, e não uma cópia redundante da mesma informação. A conclusão sobre redundância é consolidada na seção 16, com VIF e ablação.

## 11. CatBoost — challenger

O challenger usa seis configurações predefinidas, seed fixa e early stopping na Validação. Cada ajuste admite até 1.500 iterações e 100 rodadas sem melhora. Missing numérico permanece nativo e categóricas são declaradas explicitamente.

In [16]:
X_cat_treino = preparar_features_catboost(X_treino, VARIAVEIS_NUMERICAS, VARIAVEIS_CATEGORICAS)
X_cat_validacao = preparar_features_catboost(X_validacao, VARIAVEIS_NUMERICAS, VARIAVEIS_CATEGORICAS)
# Evidência histórica do tuning já concluído; não é recalculada neste fechamento.
tabela_tuning_catboost = pd.DataFrame([
    {'depth': 5, 'learning_rate': 0.03, 'best_iteration': 1153, 'roc_auc_treino': 0.8488411516, 'roc_auc_validacao': 0.8423795734, 'gini_validacao': 0.6847591469, 'ks_validacao': 0.5286561300, 'pr_auc_validacao': 0.5836961660, 'brier_validacao': 0.0759985948, 'gap_roc_auc': 0.0064615782},
    {'depth': 5, 'learning_rate': 0.05, 'best_iteration': 629, 'roc_auc_treino': 0.8481324099, 'roc_auc_validacao': 0.8427218375, 'gini_validacao': 0.6854436750, 'ks_validacao': 0.5286850428, 'pr_auc_validacao': 0.5843836886, 'brier_validacao': 0.0758994785, 'gap_roc_auc': 0.0054105724},
    {'depth': 6, 'learning_rate': 0.03, 'best_iteration': 803, 'roc_auc_treino': 0.8499099691, 'roc_auc_validacao': 0.8425955027, 'gini_validacao': 0.6851910054, 'ks_validacao': 0.5288308140, 'pr_auc_validacao': 0.5842840415, 'brier_validacao': 0.0759396703, 'gap_roc_auc': 0.0073144664},
    {'depth': 6, 'learning_rate': 0.05, 'best_iteration': 730, 'roc_auc_treino': 0.8541263540, 'roc_auc_validacao': 0.8429785923, 'gini_validacao': 0.6859571846, 'ks_validacao': 0.5301610231, 'pr_auc_validacao': 0.5839609666, 'brier_validacao': 0.0759587752, 'gap_roc_auc': 0.0111477617},
    {'depth': 7, 'learning_rate': 0.03, 'best_iteration': 673, 'roc_auc_treino': 0.8526000541, 'roc_auc_validacao': 0.8430369145, 'gini_validacao': 0.6860738290, 'ks_validacao': 0.5312723740, 'pr_auc_validacao': 0.5842301457, 'brier_validacao': 0.0759090963, 'gap_roc_auc': 0.0095631396},
    {'depth': 7, 'learning_rate': 0.05, 'best_iteration': 346, 'roc_auc_treino': 0.8506951619, 'roc_auc_validacao': 0.8426060880, 'gini_validacao': 0.6852121761, 'ks_validacao': 0.5299868101, 'pr_auc_validacao': 0.5832749326, 'brier_validacao': 0.0760059287, 'gap_roc_auc': 0.0080890739},
])
configuracao_catboost = (5, 0.05)
modelo_catboost = CatBoostClassifier(iterations=1500, depth=5, learning_rate=0.05,
    loss_function='Logloss', eval_metric='AUC', random_seed=42, verbose=False,
    allow_writing_files=False, thread_count=-1)
modelo_catboost.fit(X_cat_treino, y_treino, cat_features=VARIAVEIS_CATEGORICAS,
    eval_set=(X_cat_validacao, y_validacao), early_stopping_rounds=100, verbose=False)
prob_cat_treino = modelo_catboost.predict_proba(X_cat_treino)[:, 1]
prob_cat_validacao = modelo_catboost.predict_proba(X_cat_validacao)[:, 1]
display(tabela_tuning_catboost.style.format({
    coluna: '{:.4f}' for coluna in tabela_tuning_catboost.columns
    if coluna not in ['depth', 'best_iteration']
}))
print(f'CatBoost escolhido: depth={configuracao_catboost[0]}, learning_rate={configuracao_catboost[1]}, '
      f'best_iteration={modelo_catboost.get_best_iteration()}.')

,depth,learning_rate,best_iteration,roc_auc_treino,roc_auc_validacao,gini_validacao,ks_validacao,pr_auc_validacao,brier_validacao,gap_roc_auc
0,5,0.0300,1153,0.8488,0.8424,0.6848,0.5287,0.5837,0.0760,0.0065
1,5,0.0500,629,0.8481,0.8427,0.6854,0.5287,0.5844,0.0759,0.0054
2,6,0.0300,803,0.8499,0.8426,0.6852,0.5288,0.5843,0.0759,0.0073
3,6,0.0500,730,0.8541,0.8430,0.6860,0.5302,0.5840,0.0760,0.0111
4,7,0.0300,673,0.8526,0.8430,0.6861,0.5313,0.5842,0.0759,0.0096
5,7,0.0500,346,0.8507,0.8426,0.6852,0.5300,0.5833,0.0760,0.0081


CatBoost escolhido: depth=5, learning_rate=0.05, best_iteration=629.


In [17]:
depth_escolhida, taxa_escolhida = configuracao_catboost
linha_escolhida = tabela_tuning_catboost.query(
    'depth == @depth_escolhida and learning_rate == @taxa_escolhida').iloc[0]
auc_minimo = float(tabela_tuning_catboost['roc_auc_validacao'].min())
auc_maximo = float(tabela_tuning_catboost['roc_auc_validacao'].max())
display(Markdown(
    f"**Comentário Técnico — sensibilidade do tuning:** as seis configurações produziram ROC-AUC de "
    f"Validação entre **{auc_minimo:.4f}** e **{auc_maximo:.4f}**, uma amplitude total de "
    f"**{auc_maximo - auc_minimo:.4f}**. Nessa região o modelo mostrou **baixa sensibilidade aos "
    f"hiperparâmetros**. A escolha de `depth={depth_escolhida}` e `learning_rate={taxa_escolhida:g}` foi "
    f"feita principalmente por **simplicidade** (menor profundidade entre as equivalentes), **menor gap "
    f"Treino-Validação** (**{linha_escolhida['gap_roc_auc']:.4f}**) e **desempenho equivalente**. "
    "**Não** se afirma que essa configuração seja estatisticamente superior às demais. "
    f"O early stopping atuou dentro do orçamento: `best_iteration={int(linha_escolhida['best_iteration'])}` "
    "contra um limite de 1.500 iterações, o que indica que o critério de parada não ficou saturado."
))

**Comentário Técnico — sensibilidade do tuning:** as seis configurações produziram ROC-AUC de Validação entre **0.8424** e **0.8430**, uma amplitude total de **0.0007**. Nessa região o modelo mostrou **baixa sensibilidade aos hiperparâmetros**. A escolha de `depth=5` e `learning_rate=0.05` foi feita principalmente por **simplicidade** (menor profundidade entre as equivalentes), **menor gap Treino-Validação** (**0.0054**) e **desempenho equivalente**. **Não** se afirma que essa configuração seja estatisticamente superior às demais. O early stopping atuou dentro do orçamento: `best_iteration=629` contra um limite de 1.500 iterações, o que indica que o critério de parada não ficou saturado.

## 12. Comparação dos modelos

A comparação considera discriminação, calibração, diferença Treino-Validação e interpretabilidade. Acurácia não é utilizada.

Além dos valores pontuais, a diferença de ROC-AUC entre os modelos recebe um intervalo de confiança de 95% por **bootstrap pareado** na Validação, para dimensionar o ruído amostral.

In [18]:
probabilidades = {
    ('Regressão Logística', 'Treino'): (y_treino, prob_log_treino),
    ('Regressão Logística', 'Validação'): (y_validacao, prob_log_validacao),
    ('CatBoost', 'Treino'): (y_treino, prob_cat_treino),
    ('CatBoost', 'Validação'): (y_validacao, prob_cat_validacao),
}
registros_metricas = []
for (modelo, amostra), (alvo, probabilidade) in probabilidades.items():
    registros_metricas.append({'modelo': modelo, 'amostra': amostra,
                               **calcular_metricas_classificacao(alvo, probabilidade)})
tabela_metricas = pd.DataFrame(registros_metricas)
display(tabela_metricas.style.format({m: '{:.4f}' for m in ['roc_auc', 'gini', 'ks', 'pr_auc', 'brier']}).hide(axis='index'))
assert set(tabela_metricas['amostra']) == {'Treino', 'Validação'}
tabela_gaps = (tabela_metricas.pivot(index='modelo', columns='amostra', values=['roc_auc', 'brier'])
               .reset_index())
display(tabela_gaps)

def intervalo_delta_auc(alvo, probabilidade_referencia, probabilidade_alternativa,
                        reamostragens: int = 400, semente: int = SEMENTE_ALEATORIA):
    """Bootstrap pareado da diferença de ROC-AUC entre duas configurações.

    O gerador é reiniciado com a mesma semente a cada chamada, portanto todas as
    comparações usam exatamente as mesmas reamostragens. Serve como referência da
    escala amostral na Validação e não como teste de hipótese formal.
    """

    alvo = np.asarray(alvo)
    referencia = np.asarray(probabilidade_referencia, dtype=float)
    alternativa = np.asarray(probabilidade_alternativa, dtype=float)
    gerador = np.random.default_rng(semente)
    diferencas = []
    for _ in range(reamostragens):
        indices = gerador.integers(0, len(alvo), len(alvo))
        alvo_reamostrado = alvo[indices]
        if alvo_reamostrado.min() == alvo_reamostrado.max():
            continue
        diferencas.append(roc_auc_score(alvo_reamostrado, alternativa[indices])
                          - roc_auc_score(alvo_reamostrado, referencia[indices]))
    diferencas = np.asarray(diferencas)
    return float(np.percentile(diferencas, 2.5)), float(np.percentile(diferencas, 97.5))

limite_inferior_modelos, limite_superior_modelos = intervalo_delta_auc(
    y_validacao, prob_log_validacao, prob_cat_validacao)
print(f'Delta ROC-AUC CatBoost menos Logistica na Validacao: '
      f'IC95% [{limite_inferior_modelos:+.4f}; {limite_superior_modelos:+.4f}]')

modelo,amostra,roc_auc,gini,ks,pr_auc,brier
Regressão Logística,Treino,0.8160,0.6320,0.4917,0.4730,0.0803
Regressão Logística,Validação,0.8188,0.6376,0.4884,0.5344,0.0807
CatBoost,Treino,0.8481,0.6963,0.5429,0.5556,0.0730
CatBoost,Validação,0.8427,0.6854,0.5287,0.5844,0.0759


modelo   roc_auc               brier          
amostra                         Treino Validação    Treino Validação
0                   CatBoost  0.848132  0.842722  0.072983  0.075899
1        Regressão Logística  0.816004  0.818819  0.080272  0.080747

Delta ROC-AUC CatBoost menos Logistica na Validacao: IC95% [+0.0202; +0.0279]


In [19]:
fig_roc = go.Figure()
for modelo, probabilidade, cor in [('Regressão Logística', prob_log_validacao, CORES['principal']),
                                     ('CatBoost', prob_cat_validacao, CORES['destaque'])]:
    fpr, tpr, _ = roc_curve(y_validacao, probabilidade)
    fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=modelo, line=dict(color=cor)))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Aleatório',
                             line=dict(color=CORES['cinza'], dash='dash')))
aplicar_layout_executivo(fig_roc, 'Curva ROC na Validação', titulo_eixo_x='Taxa de falso positivo',
                         titulo_eixo_y='Taxa de verdadeiro positivo')
fig_roc.show()
salvar_grafico(fig_roc, 'final_05_roc_validacao', PASTA_FIGURAS)

fig_calibracao = go.Figure()
for modelo, probabilidade, cor in [('Regressão Logística', prob_log_validacao, CORES['principal']),
                                     ('CatBoost', prob_cat_validacao, CORES['destaque'])]:
    observado, previsto = calibration_curve(y_validacao, probabilidade, n_bins=10, strategy='quantile')
    fig_calibracao.add_trace(go.Scatter(x=previsto, y=observado, mode='lines+markers', name=modelo,
                                         line=dict(color=cor)))
fig_calibracao.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Calibração perfeita',
                                    line=dict(color=CORES['cinza'], dash='dash')))
aplicar_layout_executivo(fig_calibracao, 'Calibração na Validação',
                         titulo_eixo_x='Probabilidade média prevista', titulo_eixo_y='Taxa observada')
aplicar_eixo_percentual(fig_calibracao, 'x'); aplicar_eixo_percentual(fig_calibracao, 'y')
fig_calibracao.show()
salvar_grafico(fig_calibracao, 'final_06_calibracao_validacao', PASTA_FIGURAS)

lifts = []
for modelo, probabilidade in [('Regressão Logística', prob_log_validacao), ('CatBoost', prob_cat_validacao)]:
    lifts.append(calcular_lift_por_decil(y_validacao, probabilidade).assign(modelo=modelo))
tabela_lift = pd.concat(lifts, ignore_index=True)
fig_lift = px.line(tabela_lift, x='decil', y='captura_acumulada', color='modelo', markers=True)
aplicar_layout_executivo(fig_lift, 'Captura acumulada de eventos na Validação',
                         titulo_eixo_x='Decil de risco', titulo_eixo_y='Captura acumulada')
aplicar_eixo_percentual(fig_lift)
fig_lift.show()
salvar_grafico(fig_lift, 'final_07_captura_validacao', PASTA_FIGURAS)

In [20]:
validacao_metricas = tabela_metricas.query("amostra == 'Validação'").set_index('modelo')
gap_auc_cat = (tabela_metricas.query("modelo == 'CatBoost' and amostra == 'Treino'")['roc_auc'].iloc[0]
               - validacao_metricas.loc['CatBoost', 'roc_auc'])
gap_auc_log = (tabela_metricas.query("modelo == 'Regressão Logística' and amostra == 'Treino'")['roc_auc'].iloc[0]
               - validacao_metricas.loc['Regressão Logística', 'roc_auc'])
fig_comparacao = px.bar(tabela_metricas, x='modelo', y='roc_auc', color='amostra', barmode='group',
                         color_discrete_map={'Treino': CORES['secundaria'], 'Validação': CORES['principal']})
aplicar_layout_executivo(fig_comparacao, 'ROC-AUC por modelo e amostra',
                         titulo_eixo_x='Modelo', titulo_eixo_y='ROC-AUC')
fig_comparacao.update_yaxes(range=[0, 0.9])
fig_comparacao.show()
salvar_grafico(fig_comparacao, 'final_08_comparacao_modelos', PASTA_FIGURAS)
display(Markdown(
    f"**Análise/Interpretação:** na Validação, o CatBoost apresenta ROC-AUC "
    f"**{validacao_metricas.loc['CatBoost', 'roc_auc']:.4f}**, KS **{validacao_metricas.loc['CatBoost', 'ks']:.4f}** "
    f"e Brier **{validacao_metricas.loc['CatBoost', 'brier']:.4f}**, contra ROC-AUC "
    f"**{validacao_metricas.loc['Regressão Logística', 'roc_auc']:.4f}**, KS "
    f"**{validacao_metricas.loc['Regressão Logística', 'ks']:.4f}** e Brier "
    f"**{validacao_metricas.loc['Regressão Logística', 'brier']:.4f}** da Logística. O gap de AUC "
    f"Treino-Validação é **{gap_auc_cat:.4f}** no CatBoost e **{gap_auc_log:.4f}** na Logística. "
    f"O bootstrap pareado da diferença entre modelos dá IC95% "
    f"**[{limite_inferior_modelos:+.4f}; {limite_superior_modelos:+.4f}]**, que não contém zero."
))
display(Markdown(
    "**Champion provisório para validação final: CatBoost.** A indicação considera melhora consistente em "
    "ROC-AUC, Gini, KS, PR-AUC e Brier na Validação, com gap Treino-Validação moderado. A Regressão "
    "Logística permanece benchmark de governança e interpretabilidade. "
    "**Ressalva (seção 9):** o CatBoost recebeu early stopping e seleção de configuração na própria "
    "Validação, ajustes que a Logística não recebeu, portanto a vantagem medida carrega algum otimismo. "
    "A magnitude da diferença é bem superior ao otimismo esperado desse mecanismo, mas a confirmação "
    "independente só ocorrerá no OOT. A decisão é provisória, sujeita à revisão humana, e não usa OOT."
))

**Análise/Interpretação:** na Validação, o CatBoost apresenta ROC-AUC **0.8427**, KS **0.5287** e Brier **0.0759**, contra ROC-AUC **0.8188**, KS **0.4884** e Brier **0.0807** da Logística. O gap de AUC Treino-Validação é **0.0054** no CatBoost e **-0.0028** na Logística. O bootstrap pareado da diferença entre modelos dá IC95% **[+0.0202; +0.0279]**, que não contém zero.

**Champion provisório para validação final: CatBoost.** A indicação considera melhora consistente em ROC-AUC, Gini, KS, PR-AUC e Brier na Validação, com gap Treino-Validação moderado. A Regressão Logística permanece benchmark de governança e interpretabilidade. **Ressalva (seção 9):** o CatBoost recebeu early stopping e seleção de configuração na própria Validação, ajustes que a Logística não recebeu, portanto a vantagem medida carrega algum otimismo. A magnitude da diferença é bem superior ao otimismo esperado desse mecanismo, mas a confirmação independente só ocorrerá no OOT. A decisão é provisória, sujeita à revisão humana, e não usa OOT.

## 13. Robustez de features — ablações controladas

**Pergunta:** quanto cada feature sob suspeita contribui de fato, e essa contribuição é distinguível de ruído amostral?

Todos os cenários reutilizam a configuração CatBoost já congelada e alteram somente o conjunto de features. A avaliação usa exclusivamente a Validação. Além do delta pontual de ROC-AUC, cada cenário recebe um **intervalo de confiança de 95% por bootstrap pareado**, que dá a escala do ruído amostral e evita decidir por diferenças na terceira casa decimal.

In [21]:
cenarios_ablacao = {
    'Completo': [],
    'Sem cat_var10': ['cat_var10'],
    'Sem cat_var13': ['cat_var13'],
    'Sem cat_var10 e cat_var13': ['cat_var10', 'cat_var13'],
    'Sem cat_var6': ['cat_var6'],
    'Sem var1': ['var1'],
    'Sem var3': ['var3'],
    'Sem var1 e var3': ['var1', 'var3'],
}
resultados_ablacao = []
probabilidades_ablacao = {}
for cenario, removidas in cenarios_ablacao.items():
    if cenario == 'Completo':
        probabilidade = prob_cat_validacao
    else:
        numericas = [v for v in VARIAVEIS_NUMERICAS if v not in removidas]
        categoricas = [v for v in VARIAVEIS_CATEGORICAS if v not in removidas]
        X_tr = preparar_features_catboost(X_treino.drop(columns=removidas), numericas, categoricas)
        X_va = preparar_features_catboost(X_validacao.drop(columns=removidas), numericas, categoricas)
        modelo = CatBoostClassifier(iterations=1500, depth=configuracao_catboost[0],
            learning_rate=configuracao_catboost[1], loss_function='Logloss', eval_metric='AUC',
            random_seed=SEMENTE_ALEATORIA, verbose=False, allow_writing_files=False, thread_count=-1)
        modelo.fit(X_tr, y_treino, cat_features=categoricas, eval_set=(X_va, y_validacao),
                   early_stopping_rounds=100, verbose=False)
        probabilidade = modelo.predict_proba(X_va)[:, 1]
    probabilidades_ablacao[cenario] = probabilidade
    resultados_ablacao.append({'cenario': cenario,
        **calcular_metricas_classificacao(y_validacao, probabilidade)})
tabela_ablacao = pd.DataFrame(resultados_ablacao)
auc_completo = tabela_ablacao.set_index('cenario').loc['Completo', 'roc_auc']
tabela_ablacao['delta_roc_auc_vs_completo'] = tabela_ablacao['roc_auc'] - auc_completo
limites_ablacao = []
for cenario in tabela_ablacao['cenario']:
    if cenario == 'Completo':
        limites_ablacao.append((0.0, 0.0))
    else:
        limites_ablacao.append(
            intervalo_delta_auc(y_validacao, prob_cat_validacao, probabilidades_ablacao[cenario]))
tabela_ablacao['delta_ic95_inferior'] = [x[0] for x in limites_ablacao]
tabela_ablacao['delta_ic95_superior'] = [x[1] for x in limites_ablacao]
tabela_ablacao['contribuicao_distinguivel'] = np.where(
    tabela_ablacao['cenario'].eq('Completo'), '-',
    np.where((tabela_ablacao['delta_ic95_inferior'] < 0) & (tabela_ablacao['delta_ic95_superior'] < 0),
             'sim', 'nao'))
display(tabela_ablacao.style.format({c: '{:.4f}' for c in tabela_ablacao.columns
                                     if c not in ['cenario', 'contribuicao_distinguivel']}).hide(axis='index'))
display(Markdown(
    "**Comentário Técnico:** `contribuicao_distinguivel = sim` significa que o intervalo de 95% do delta de "
    "ROC-AUC ficou inteiramente abaixo de zero, ou seja, remover a feature piora o modelo de forma "
    "distinguível do ruído amostral na Validação. `nao` significa que o intervalo contém zero e a "
    "contribuição incremental **não** é separável do ruído. O bootstrap é pareado e usa as mesmas "
    "reamostragens em todos os cenários. Ele mede escala amostral dentro da Validação e não substitui a "
    "avaliação fora do tempo."
))

cenario,roc_auc,gini,ks,pr_auc,brier,delta_roc_auc_vs_completo,delta_ic95_inferior,delta_ic95_superior,contribuicao_distinguivel
Completo,0.8427,0.6854,0.5287,0.5844,0.0759,0.0000,0.0000,0.0000,-
Sem cat_var10,0.8406,0.6812,0.5247,0.5808,0.0762,-0.0021,-0.0032,-0.0011,sim
Sem cat_var13,0.8415,0.6830,0.5294,0.5827,0.0761,-0.0012,-0.0020,-0.0003,sim
Sem cat_var10 e cat_var13,0.8396,0.6792,0.5249,0.5815,0.0762,-0.0031,-0.0045,-0.0017,sim
Sem cat_var6,0.8424,0.6847,0.5286,0.5832,0.0761,-0.0004,-0.0011,0.0004,nao
Sem var1,0.8017,0.6034,0.4672,0.4199,0.0908,-0.0410,-0.0450,-0.0373,sim
Sem var3,0.8297,0.6594,0.4979,0.5502,0.0794,-0.0130,-0.0154,-0.0109,sim
Sem var1 e var3,0.7939,0.5878,0.4505,0.3919,0.0926,-0.0488,-0.0530,-0.0449,sim


**Comentário Técnico:** `contribuicao_distinguivel = sim` significa que o intervalo de 95% do delta de ROC-AUC ficou inteiramente abaixo de zero, ou seja, remover a feature piora o modelo de forma distinguível do ruído amostral na Validação. `nao` significa que o intervalo contém zero e a contribuição incremental **não** é separável do ruído. O bootstrap é pareado e usa as mesmas reamostragens em todos os cenários. Ele mede escala amostral dentro da Validação e não substitui a avaliação fora do tempo.

## 14. `var12` — representação híbrida versus categórica

**Pergunta:** a componente contínua de `var12` acrescenta algo além do que os códigos já informam?

Comparam-se duas representações sob a mesma configuração CatBoost congelada:

- **A — híbrida (atual):** componente contínua mascarada e imputada, mais três indicadores binários;
- **B — categórica simplificada:** um único estado com valores `99997`, `99998`, `99999` e `REGULAR`, com estado próprio para ausência nativa caso ela exista.

A representação A é a referência comparativa inicial desta análise de desenvolvimento. A decisão final pré-OOT entre A e B é registrada nas seções seguintes e não é tomada automaticamente pelo ROC-AUC.

In [22]:
def preparar_var12_categorico(dados: pd.DataFrame, variaveis_numericas: list[str],
                               variaveis_categoricas: list[str]) -> pd.DataFrame:
    """Representa ``var12`` como um único estado categórico.

    Estados possíveis: ``99997``, ``99998``, ``99999``, ``REGULAR`` e
    ``MISSING``. A ausência nativa recebe estado próprio e nunca é misturada
    silenciosamente aos códigos especiais.
    """

    numericas_restantes = [v for v in variaveis_numericas if v != 'var12']
    resultado = dados[numericas_restantes + variaveis_categoricas].copy()
    resultado['var12_estado'] = criar_var12_estado(dados['var12'])
    for variavel in variaveis_categoricas:
        resultado[variavel] = resultado[variavel].astype('string').fillna('__MISSING__')
    return resultado

X_var12cat_treino = preparar_var12_categorico(X_treino, VARIAVEIS_NUMERICAS, VARIAVEIS_CATEGORICAS)
X_var12cat_validacao = preparar_var12_categorico(X_validacao, VARIAVEIS_NUMERICAS, VARIAVEIS_CATEGORICAS)
categoricas_var12 = VARIAVEIS_CATEGORICAS + ['var12_estado']
modelo_var12_categorico = CatBoostClassifier(iterations=1500, depth=configuracao_catboost[0],
    learning_rate=configuracao_catboost[1], loss_function='Logloss', eval_metric='AUC',
    random_seed=SEMENTE_ALEATORIA, verbose=False, allow_writing_files=False, thread_count=-1)
modelo_var12_categorico.fit(X_var12cat_treino, y_treino, cat_features=categoricas_var12,
    eval_set=(X_var12cat_validacao, y_validacao), early_stopping_rounds=100, verbose=False)
prob_var12_categorico = modelo_var12_categorico.predict_proba(X_var12cat_validacao)[:, 1]

comparacao_var12 = pd.DataFrame([
    {'representacao': 'A - híbrida (contínua + 3 indicadores)', 'colunas_de_var12': 4,
     **calcular_metricas_classificacao(y_validacao, prob_cat_validacao)},
    {'representacao': 'B - categórica (99997/99998/99999/REGULAR/MISSING)', 'colunas_de_var12': 1,
     **calcular_metricas_classificacao(y_validacao, prob_var12_categorico)},
])
comparacao_var12['delta_roc_auc'] = comparacao_var12['roc_auc'] - comparacao_var12['roc_auc'].iloc[0]
limite_inferior_var12, limite_superior_var12 = intervalo_delta_auc(
    y_validacao, prob_cat_validacao, prob_var12_categorico)
display(comparacao_var12.style.format({c: '{:.4f}' for c in comparacao_var12.columns
                                       if c not in ['representacao', 'colunas_de_var12']}).hide(axis='index'))
estados_observados = sorted(X_var12cat_treino['var12_estado'].dropna().unique().tolist())
equivalentes_var12 = limite_inferior_var12 < 0 < limite_superior_var12
display(Markdown(
    f"**Análise/Interpretação:** os estados observados no Treino são **{estados_observados}**. Não há "
    "ausência nativa em `var12`, portanto o estado `MISSING` não aparece nesta base, mas está previsto "
    "na função para que uma ausência futura não seja confundida com código especial. "
    f"O delta de ROC-AUC de B em relação a A é **{comparacao_var12.iloc[1]['delta_roc_auc']:+.4f}**, com "
    f"IC95% pareado **[{limite_inferior_var12:+.4f}; {limite_superior_var12:+.4f}]**, que "
    f"**{'contém' if equivalentes_var12 else 'não contém'}** zero. "
    f"{'As duas representações são essencialmente equivalentes em desempenho' if equivalentes_var12 else 'As representações diferem de forma distinguível do ruído'}. "
    "A representação B usa **uma** coluna no lugar de quatro e dispensa imputador e padronizador para uma "
    "componente contínua observada em apenas 6% dos registros, o que é uma vantagem de simplicidade e de "
    "manutenção."
))
display(Markdown(
    "**Decisão humana aprovada:** utilizar a representação **B**, `var12_estado`, no candidato final pré-OOT. "
    "O desempenho é equivalente à representação híbrida, com menor complexidade e maior interpretabilidade. "
    "A representação A permanece preservada como evidência de desenvolvimento; nenhuma escolha utiliza OOT."
))

representacao,colunas_de_var12,roc_auc,gini,ks,pr_auc,brier,delta_roc_auc
A - híbrida (contínua + 3 indicadores),4,0.8427,0.6854,0.5287,0.5844,0.0759,0.0000
B - categórica (99997/99998/99999/REGULAR/MISSING),1,0.8424,0.6848,0.5271,0.5838,0.0759,-0.0003


**Análise/Interpretação:** os estados observados no Treino são **['99997', '99998', '99999', 'REGULAR']**. Não há ausência nativa em `var12`, portanto o estado `MISSING` não aparece nesta base, mas está previsto na função para que uma ausência futura não seja confundida com código especial. O delta de ROC-AUC de B em relação a A é **-0.0003**, com IC95% pareado **[-0.0009; +0.0003]**, que **contém** zero. As duas representações são essencialmente equivalentes em desempenho. A representação B usa **uma** coluna no lugar de quatro e dispensa imputador e padronizador para uma componente contínua observada em apenas 6% dos registros, o que é uma vantagem de simplicidade e de manutenção.

**Decisão humana aprovada:** utilizar a representação **B**, `var12_estado`, no candidato final pré-OOT. O desempenho é equivalente à representação híbrida, com menor complexidade e maior interpretabilidade. A representação A permanece preservada como evidência de desenvolvimento; nenhuma escolha utiliza OOT.

## 15. Explicabilidade em desenvolvimento

**Comentário Técnico:** as leituras descrevem associação, direção e magnitude. Não são causais e não atribuem significado de negócio às variáveis anonimizadas.

Os coeficientes da Regressão Logística são apresentados em **blocos separados** porque as escalas não são comparáveis entre si: variáveis numéricas foram padronizadas e o coeficiente representa o efeito por desvio-padrão, enquanto indicadores binários e dummies categóricas não foram padronizados e o coeficiente representa o efeito da transição de 0 para 1. Um ranking único entre blocos seria enganoso.

O SHAP desta seção é calculado sobre o modelo com o **conjunto completo** de features, porque é essa a informação necessária para decidir, na seção 17, quais features permanecem.

In [23]:
coeficientes_logistica = pd.DataFrame({'feature_transformada':
    pipeline_logistica.named_steps['preprocessamento'].get_feature_names_out(),
    'coeficiente': pipeline_logistica.named_steps['modelo'].coef_[0]})

def classificar_bloco(nome: str) -> str:
    if 'missingindicator' in nome or nome.startswith('var12_indicadores__'):
        return '2. Indicadores binários (transição 0 para 1)'
    if nome.startswith('categoricas__'):
        return '3. Dummies categóricas (transição 0 para 1)'
    return '1. Numéricas padronizadas (efeito por desvio-padrão)'

coeficientes_logistica['bloco'] = coeficientes_logistica['feature_transformada'].map(classificar_bloco)
coeficientes_logistica['magnitude'] = coeficientes_logistica['coeficiente'].abs()
coeficientes_logistica = coeficientes_logistica.sort_values(['bloco', 'magnitude'], ascending=[True, False])
for bloco, grupo in coeficientes_logistica.groupby('bloco'):
    display(Markdown(f'**{bloco}** — magnitudes comparáveis apenas dentro deste bloco'))
    display(grupo.head(10)[['feature_transformada', 'coeficiente']]
            .style.format({'coeficiente': '{:+.4f}'}).hide(axis='index'))
display(Markdown('**Componentes de `var12` na Logística**'))
display(coeficientes_logistica[coeficientes_logistica['feature_transformada'].str.contains('var12')]
        [['bloco', 'feature_transformada', 'coeficiente']]
        .style.format({'coeficiente': '{:+.4f}'}).hide(axis='index'))
display(Markdown(
    "**Análise/Interpretação:** as magnitudes **não** devem ser comparadas entre blocos. Um coeficiente de "
    "0,7 em um indicador binário que atinge metade da população não equivale a um coeficiente de 0,7 em "
    "uma variável padronizada, cujo efeito é medido por desvio-padrão. Dentro do bloco 1, `var1` e `var3` "
    "aparecem com sinais opostos e magnitudes próximas, coerente com o diagnóstico de contraste da "
    "seção 16. A regularização L2 mantém os coeficientes finitos por construção, portanto a ausência de "
    "coeficientes explosivos **não** é, por si, evidência contra colinearidade: essa questão é respondida "
    "pelo VIF, pela estabilidade de sinal em `C` e pelas ablações. Nenhuma leitura aqui é causal."
))

**1. Numéricas padronizadas (efeito por desvio-padrão)** — magnitudes comparáveis apenas dentro deste bloco

feature_transformada,coeficiente
numericas__var1,-1.1891
numericas__var3,+0.9734
numericas__var11,-0.1937
numericas__var7,+0.1778
numericas__var14,-0.1482
numericas__var9,-0.0877
numericas__var5,-0.0632
var12_continua__var12_continua,+0.0315
numericas__var4,+0.0206
numericas__var8,+0.0086


**2. Indicadores binários (transição 0 para 1)** — magnitudes comparáveis apenas dentro deste bloco

feature_transformada,coeficiente
var12_indicadores__var12_codigo_99999,+0.6938
var12_indicadores__var12_codigo_99998,+0.6425
var12_indicadores__var12_codigo_99997,-0.0420
numericas__missingindicator_var11,-0.0389
numericas__missingindicator_var4,-0.0370
numericas__missingindicator_var9,-0.0198
numericas__missingindicator_var8,+0.0039


**3. Dummies categóricas (transição 0 para 1)** — magnitudes comparáveis apenas dentro deste bloco

feature_transformada,coeficiente
categoricas__cat_var2_1.0,-0.9823
categoricas__cat_var10_0,-0.6456
categoricas__cat_var15_12.0,-0.5391
categoricas__cat_var13_10.0,-0.4930
categoricas__cat_var13_12.0,+0.4620
categoricas__cat_var2___MISSING__,-0.3595
categoricas__cat_var2_0.333333333333333,+0.3252
categoricas__cat_var13_2.0,-0.2908
categoricas__cat_var13_3.0,-0.2841
categoricas__cat_var13_1.0,-0.2764


**Componentes de `var12` na Logística**

bloco,feature_transformada,coeficiente
1. Numéricas padronizadas (efeito por desvio-padrão),var12_continua__var12_continua,+0.0315
2. Indicadores binários (transição 0 para 1),var12_indicadores__var12_codigo_99999,+0.6938
2. Indicadores binários (transição 0 para 1),var12_indicadores__var12_codigo_99998,+0.6425
2. Indicadores binários (transição 0 para 1),var12_indicadores__var12_codigo_99997,-0.0420


**Análise/Interpretação:** as magnitudes **não** devem ser comparadas entre blocos. Um coeficiente de 0,7 em um indicador binário que atinge metade da população não equivale a um coeficiente de 0,7 em uma variável padronizada, cujo efeito é medido por desvio-padrão. Dentro do bloco 1, `var1` e `var3` aparecem com sinais opostos e magnitudes próximas, coerente com o diagnóstico de contraste da seção 16. A regularização L2 mantém os coeficientes finitos por construção, portanto a ausência de coeficientes explosivos **não** é, por si, evidência contra colinearidade: essa questão é respondida pelo VIF, pela estabilidade de sinal em `C` e pelas ablações. Nenhuma leitura aqui é causal.

In [24]:
amostra_shap = X_cat_validacao.sample(n=min(3000, len(X_cat_validacao)), random_state=SEMENTE_ALEATORIA)
valores_shap = modelo_catboost.get_feature_importance(
    Pool(amostra_shap, cat_features=VARIAVEIS_CATEGORICAS), type='ShapValues')[:, :-1]
tabela_shap = pd.DataFrame({'feature': amostra_shap.columns,
    'shap_abs_medio': np.abs(valores_shap).mean(axis=0), 'shap_medio': valores_shap.mean(axis=0)
}).sort_values('shap_abs_medio', ascending=False)
top10_shap = tabela_shap.head(10).sort_values('shap_abs_medio')
fig_shap = px.bar(top10_shap, x='shap_abs_medio', y='feature', orientation='h',
    color_discrete_sequence=[CORES['principal']])
aplicar_layout_executivo(fig_shap, 'Top 10 features por importância SHAP média na Validação',
    titulo_eixo_x='Média de |SHAP|', titulo_eixo_y='Feature', mostrar_legenda=False)
fig_shap.show(); salvar_grafico(fig_shap, 'final_09_shap_global', PASTA_FIGURAS)
display(tabela_shap.head(10).style.format({'shap_abs_medio': '{:.4f}',
                                           'shap_medio': '{:+.4f}'}).hide(axis='index'))
dependencia_monitorada = tabela_shap.set_index('feature').reindex(['cat_var6', 'cat_var10', 'cat_var13',
    'var12', 'var12_codigo_99997', 'var12_codigo_99998', 'var12_codigo_99999']).dropna().reset_index()
display(Markdown('**Dependência monitorada: cat_var6, cat_var10, cat_var13 e componentes de var12**'))
display(dependencia_monitorada.style.format({'shap_abs_medio': '{:.4f}',
                                             'shap_medio': '{:+.4f}'}).hide(axis='index'))
detalhes_shap = []
for feature in tabela_shap.head(3)['feature']:
    posicao = amostra_shap.columns.get_loc(feature); serie = amostra_shap[feature]
    grupo = (serie.astype('string').fillna('__MISSING__') if feature in VARIAVEIS_CATEGORICAS
             else pd.qcut(serie.rank(method='first'), 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']))
    resumo = pd.DataFrame({'grupo': grupo, 'shap': valores_shap[:, posicao]}).groupby(
        'grupo', observed=True).agg(shap_medio=('shap', 'mean'), registros=('shap', 'size')).reset_index()
    resumo.insert(0, 'feature', feature); detalhes_shap.append(resumo)
tabela_detalhes_shap = pd.concat(detalhes_shap, ignore_index=True)
display(tabela_detalhes_shap.style.format({'shap_medio': '{:+.4f}'}).hide(axis='index'))
maior_shap = tabela_shap.iloc[0]
display(Markdown(
    f"**Análise/Interpretação:** a feature de maior importância média é **`{maior_shap['feature']}`**, com "
    f"|SHAP| médio de **{maior_shap['shap_abs_medio']:.4f}**. A importância não está concentrada em uma "
    "única variável. Os componentes de `var12` aparecem separados, o que permite verificar que o efeito "
    "dos códigos especiais não está diluído na componente contínua. A leitura por quintis mostra direções "
    "monotônicas nas principais numéricas, com `var1` e `var3` em sentidos opostos. Nenhum significado de "
    "negócio é atribuído às variáveis anonimizadas."
))

feature,shap_abs_medio,shap_medio
var1,0.4823,-0.1551
var3,0.4553,+0.1310
cat_var2,0.2313,+0.0056
var5,0.2193,+0.0073
cat_var10,0.2039,+0.0933
cat_var6,0.1116,+0.0210
var7,0.1113,+0.0184
var14,0.1050,+0.0019
var11,0.0839,-0.0088
var9,0.0831,+0.0104


**Dependência monitorada: cat_var6, cat_var10, cat_var13 e componentes de var12**

feature,shap_abs_medio,shap_medio
cat_var6,0.1116,+0.0210
cat_var10,0.2039,+0.0933
cat_var13,0.0680,-0.0009
var12,0.0136,-0.0012
var12_codigo_99997,0.0165,+0.0024
var12_codigo_99998,0.0047,-0.0025
var12_codigo_99999,0.0445,-0.0069


feature,grupo,shap_medio,registros
var1,Q1,+0.7252,600
var1,Q2,+0.0616,600
var1,Q3,-0.2320,600
var1,Q4,-0.5148,600
var1,Q5,-0.8153,600
var3,Q1,-0.5501,600
var3,Q2,-0.2469,600
var3,Q3,+0.1465,600
var3,Q4,+0.4667,600
var3,Q5,+0.8389,600


**Análise/Interpretação:** a feature de maior importância média é **`var1`**, com |SHAP| médio de **0.4823**. A importância não está concentrada em uma única variável. Os componentes de `var12` aparecem separados, o que permite verificar que o efeito dos códigos especiais não está diluído na componente contínua. A leitura por quintis mostra direções monotônicas nas principais numéricas, com `var1` e `var3` em sentidos opostos. Nenhum significado de negócio é atribuído às variáveis anonimizadas.

## 16. Diagnóstico consolidado de redundância — `var1` e `var3`

**Pergunta:** a correlação de Spearman de 0,82 entre `var1` e `var3`, somada a coeficientes de sinais opostos, representa redundância controlável ou instabilidade que ameaça a generalização?

A conclusão reúne cinco evidências independentes — correlação de postos, VIF, sinal univariado, coeficiente logístico e importância SHAP — mais as ablações da seção 13 e a estabilidade de sinal da seção 10. O critério registrado é: sinal instável sob variação de `C` **ou** VIF maior ou igual a 10 caracterizam **evidência de instabilidade**; sinal estável, VIF abaixo de 10 e contribuição distinguível das duas variáveis caracterizam **redundância controlável**; qualquer outra combinação é **evidência insuficiente**. Nenhuma feature derivada é criada automaticamente.

In [25]:
def resumir_par_redundante(variavel: str) -> dict:
    linha_features = tabela_features.set_index('variavel').loc[variavel]
    linha_vif = tabela_vif.set_index('variavel').loc[variavel]
    linha_shap = tabela_shap.set_index('feature').loc[variavel]
    linha_ablacao = tabela_ablacao.set_index('cenario').loc[f'Sem {variavel}']
    return {
        'variavel': variavel,
        'spearman_var1_var3': correlacao_spearman.loc['var1', 'var3'],
        'vif': linha_vif['vif'],
        'gini_univariado_abs': linha_features['sinal_preditivo_gini_abs'],
        'psi_treino_validacao': linha_features['estabilidade_psi'],
        'coeficiente_logistico': obter_coeficiente(pipeline_logistica, f'__{variavel}'),
        'shap_abs_medio': linha_shap['shap_abs_medio'],
        'shap_medio': linha_shap['shap_medio'],
        'ablacao_delta_roc_auc': linha_ablacao['delta_roc_auc_vs_completo'],
        'ablacao_ic95_inferior': linha_ablacao['delta_ic95_inferior'],
        'ablacao_ic95_superior': linha_ablacao['delta_ic95_superior'],
        'contribuicao_distinguivel': linha_ablacao['contribuicao_distinguivel'],
    }

tabela_par_var1_var3 = pd.DataFrame([resumir_par_redundante('var1'), resumir_par_redundante('var3')])
display(tabela_par_var1_var3.style.format(
    {c: '{:.4f}' for c in tabela_par_var1_var3.columns
     if c not in ['variavel', 'contribuicao_distinguivel']}).hide(axis='index'))
colunas_ablacao = ['cenario', 'roc_auc', 'gini', 'ks', 'pr_auc', 'brier',
                   'delta_roc_auc_vs_completo', 'delta_ic95_inferior', 'delta_ic95_superior',
                   'contribuicao_distinguivel']
display(tabela_ablacao[tabela_ablacao['cenario'].isin(
    ['Completo', 'Sem var1', 'Sem var3', 'Sem var1 e var3'])][colunas_ablacao]
    .style.format({c: '{:.4f}' for c in colunas_ablacao
                   if c not in ['cenario', 'contribuicao_distinguivel']}).hide(axis='index'))

linha_par = tabela_ablacao.set_index('cenario').loc['Sem var1 e var3']
vif_par = float(tabela_par_var1_var3['vif'].max())
ambas_distinguiveis = (tabela_par_var1_var3['contribuicao_distinguivel'] == 'sim').all()
if (not sinal_estavel) or vif_par >= 10:
    veredito_par = 'evidência de instabilidade'
elif ambas_distinguiveis:
    veredito_par = 'redundância controlável'
else:
    veredito_par = 'evidência insuficiente'
display(Markdown(
    f"**Análise/Interpretação:** o VIF máximo do par é **{vif_par:.2f}** e o sinal dos coeficientes "
    f"**{'permanece estável' if sinal_estavel else 'não permanece estável'}** com `C` variando de 0,01 a 10. "
    f"Remover `var1` custa **{tabela_par_var1_var3.iloc[0]['ablacao_delta_roc_auc']:+.4f}** de ROC-AUC "
    f"(IC95% [{tabela_par_var1_var3.iloc[0]['ablacao_ic95_inferior']:+.4f}; "
    f"{tabela_par_var1_var3.iloc[0]['ablacao_ic95_superior']:+.4f}]); remover `var3` custa "
    f"**{tabela_par_var1_var3.iloc[1]['ablacao_delta_roc_auc']:+.4f}** "
    f"(IC95% [{tabela_par_var1_var3.iloc[1]['ablacao_ic95_inferior']:+.4f}; "
    f"{tabela_par_var1_var3.iloc[1]['ablacao_ic95_superior']:+.4f}]); remover as duas custa "
    f"**{linha_par['delta_roc_auc_vs_completo']:+.4f}**. Ambas figuram entre as maiores importâncias SHAP "
    "do CatBoost, com direções opostas e monotônicas, o que indica que o modelo utiliza o **contraste** "
    "entre elas e não uma cópia redundante da mesma informação. O PSI de ambas é baixo, portanto não há "
    "instabilidade populacional associada ao par."
))
display(Markdown(
    f"**Conclusão do diagnóstico: {veredito_par}.** Nenhuma feature derivada é criada e nenhuma das duas é "
    "removida. A ressalva que permanece registrada é de interpretação, não de desempenho: os coeficientes "
    "individuais de `var1` e `var3` só devem ser lidos na presença do par, nunca isoladamente."
))

variavel,spearman_var1_var3,vif,gini_univariado_abs,psi_treino_validacao,coeficiente_logistico,shap_abs_medio,shap_medio,ablacao_delta_roc_auc,ablacao_ic95_inferior,ablacao_ic95_superior,contribuicao_distinguivel
var1,0.8220,3.1634,0.2686,0.0274,-1.1891,0.4823,-0.1551,-0.0410,-0.0450,-0.0373,sim
var3,0.8220,3.2904,0.2488,0.0322,0.9734,0.4553,0.1310,-0.0130,-0.0154,-0.0109,sim


cenario,roc_auc,gini,ks,pr_auc,brier,delta_roc_auc_vs_completo,delta_ic95_inferior,delta_ic95_superior,contribuicao_distinguivel
Completo,0.8427,0.6854,0.5287,0.5844,0.0759,0.0000,0.0000,0.0000,-
Sem var1,0.8017,0.6034,0.4672,0.4199,0.0908,-0.0410,-0.0450,-0.0373,sim
Sem var3,0.8297,0.6594,0.4979,0.5502,0.0794,-0.0130,-0.0154,-0.0109,sim
Sem var1 e var3,0.7939,0.5878,0.4505,0.3919,0.0926,-0.0488,-0.0530,-0.0449,sim


**Análise/Interpretação:** o VIF máximo do par é **3.29** e o sinal dos coeficientes **permanece estável** com `C` variando de 0,01 a 10. Remover `var1` custa **-0.0410** de ROC-AUC (IC95% [-0.0450; -0.0373]); remover `var3` custa **-0.0130** (IC95% [-0.0154; -0.0109]); remover as duas custa **-0.0488**. Ambas figuram entre as maiores importâncias SHAP do CatBoost, com direções opostas e monotônicas, o que indica que o modelo utiliza o **contraste** entre elas e não uma cópia redundante da mesma informação. O PSI de ambas é baixo, portanto não há instabilidade populacional associada ao par.

**Conclusão do diagnóstico: redundância controlável.** Nenhuma feature derivada é criada e nenhuma das duas é removida. A ressalva que permanece registrada é de interpretação, não de desempenho: os coeficientes individuais de `var1` e `var3` só devem ser lidos na presença do par, nunca isoladamente.

## 17. Decisão final de features

**Comentário Técnico:** a regra abaixo organiza o suporte quantitativo à decisão. A tabela final também incorpora julgamento conjunto de robustez; a remoção de `cat_var13` é uma exceção humana explícita, pois sua ablação foi distinguível de zero e o teste intra-safra indicou sinal próprio. Essa decisão foi congelada antes do OOT e não será reaberta com seus resultados.

Uma feature é **removida** quando ocorre pelo menos uma das duas situações:

1. **instabilidade sem contrapartida** — PSI Treino versus Validação maior ou igual a 0,10 **e** contribuição incremental não distinguível do ruído amostral, isto é, o intervalo de 95% do delta de ROC-AUC da ablação contém zero. A variável carrega risco de deterioração populacional sem entregar ganho mensurável;
2. **contribuição que é artefato do tempo** — evidência de proxy temporal no teste condicional à safra, independentemente do tamanho da contribuição. Uma contribuição explicada pela safra não sobrevive fora do tempo.

Nos demais casos a feature é mantida: `manter_com_tratamento` quando exige tratamento explícito e `manter` no restante.

A regra é deliberadamente simétrica e evita as duas falhas opostas: remover uma variável útil só porque ela é instável, e manter uma variável instável só porque o delta de AUC ficou negativo na quarta casa decimal sem significância amostral. Instabilidade baixa dispensa justificativa de contribuição, porque não há custo de deterioração a pagar.

**Nenhuma feature permanece como `avaliar`:** a decisão é fechada antes do OOT.

In [26]:
registros_decisao = []
for _, linha in tabela_features.iterrows():
    variavel = linha['variavel']
    psi = float(linha['estabilidade_psi'])
    evidencias = [f'PSI T-V={psi:.3f}']
    if f'Sem {variavel}' in set(tabela_ablacao['cenario']):
        linha_ablacao = tabela_ablacao.set_index('cenario').loc[f'Sem {variavel}']
        evidencias.append(
            f"ablação ΔAUC={linha_ablacao['delta_roc_auc_vs_completo']:+.4f} "
            f"IC95%[{linha_ablacao['delta_ic95_inferior']:+.4f};{linha_ablacao['delta_ic95_superior']:+.4f}] "
            f"distinguível={linha_ablacao['contribuicao_distinguivel']}")
    if variavel in set(tabela_shap['feature']):
        evidencias.append(f"SHAP={tabela_shap.set_index('feature').loc[variavel, 'shap_abs_medio']:.4f}")
    if variavel in set(tabela_vif['variavel']):
        evidencias.append(f"VIF={tabela_vif.set_index('variavel').loc[variavel, 'vif']:.2f}")
    if variavel == 'cat_var13':
        evidencias.append(f'consistência intra-safra={consistencia_cat13:.0%} ({evidencia_cat13})')
    if variavel == 'var12':
        evidencias.append(f"mascarado após tratamento={linha['mascarado_apos_tratamento_treino']:.1%}")
    if variavel in ('var1', 'var3'):
        evidencias.append(f'par var1/var3: {veredito_par}')

    decisao_final, motivo = 'manter', 'decisão humana baseada no conjunto de evidências'
    if variavel == 'var12':
        decisao_final, motivo = 'manter_com_tratamento', 'usar var12_estado categórica'
    elif variavel == 'cat_var6':
        decisao_final = 'remover'
        motivo = 'PSI material e contribuição incremental não demonstrada; IC95% inclui zero'
    elif variavel == 'cat_var13':
        decisao_final = 'remover'
        motivo = 'sinal existente, porém instabilidade temporal elevada em relação ao ganho incremental observado'
    elif variavel == 'cat_var10':
        motivo = 'contribuição incremental e SHAP justificam manutenção com monitoramento temporal'
    elif variavel in ('var1', 'var3'):
        motivo = 'redundância controlável; sinais estáveis, VIF moderado e contribuição material'
    registros_decisao.append({'variavel': variavel, 'tipo': linha['tipo'],
                              'decisao_final': decisao_final, 'motivo': motivo,
                              'evidencia': '; '.join(evidencias)})

tabela_decisao_features = pd.DataFrame(registros_decisao).sort_values(['decisao_final', 'variavel'])
with pd.option_context('display.max_colwidth', 200):
    display(tabela_decisao_features.style.hide(axis='index'))
assert 'avaliar' not in set(tabela_decisao_features['decisao_final'])
mantidas = set(tabela_decisao_features.query("decisao_final != 'remover'")['variavel'])
features_finais = [v for v in VARIAVEIS_MODELO if v in mantidas]
features_removidas = [v for v in VARIAVEIS_MODELO if v not in mantidas]
decisoes_por_variavel = tabela_decisao_features.set_index('variavel')['decisao_final'].to_dict()
display(tabela_decisao_features['decisao_final'].value_counts().rename_axis('decisao').to_frame('features'))

display(Markdown(
    f"**Decisão humana aprovada:** o conjunto final pré-OOT tem **{len(features_finais)} features originais**. "
    f"Removidas: **{', '.join(features_removidas) if features_removidas else 'nenhuma'}**.\n\n"
    "- `var12` será representada exclusivamente por `var12_estado`.\n"
    "- `cat_var6` foi removida por PSI material combinado a contribuição incremental não demonstrada.\n"
    "- `cat_var13` foi removida por robustez: há sinal, mas a instabilidade temporal é elevada em "
    "relação ao ganho incremental. Ela não é classificada como leakage.\n"
    "- `cat_var10` permanece com monitoramento temporal obrigatório.\n"
    "- `var1` e `var3` permanecem como redundância controlável; nenhuma feature derivada foi criada."
))

variavel,tipo,decisao_final,motivo,evidencia
cat_var10,categorica,manter,contribuição incremental e SHAP justificam manutenção com monitoramento temporal,PSI T-V=0.165; ablação ΔAUC=-0.0021 IC95%[-0.0032;-0.0011] distinguível=sim; SHAP=0.2039
cat_var15,categorica,manter,decisão humana baseada no conjunto de evidências,PSI T-V=0.017; SHAP=0.0558
cat_var2,categorica,manter,decisão humana baseada no conjunto de evidências,PSI T-V=0.002; SHAP=0.2313
var1,numerica,manter,"redundância controlável; sinais estáveis, VIF moderado e contribuição material",PSI T-V=0.027; ablação ΔAUC=-0.0410 IC95%[-0.0450;-0.0373] distinguível=sim; SHAP=0.4823; VIF=3.16; par var1/var3: redundância controlável
var11,numerica,manter,decisão humana baseada no conjunto de evidências,PSI T-V=0.006; SHAP=0.0839; VIF=3.66
var14,numerica,manter,decisão humana baseada no conjunto de evidências,PSI T-V=0.006; SHAP=0.1050; VIF=2.45
var3,numerica,manter,"redundância controlável; sinais estáveis, VIF moderado e contribuição material",PSI T-V=0.032; ablação ΔAUC=-0.0130 IC95%[-0.0154;-0.0109] distinguível=sim; SHAP=0.4553; VIF=3.29; par var1/var3: redundância controlável
var4,numerica,manter,decisão humana baseada no conjunto de evidências,PSI T-V=0.002; SHAP=0.0446; VIF=2.06
var5,numerica,manter,decisão humana baseada no conjunto de evidências,PSI T-V=0.003; SHAP=0.2193; VIF=6.52
var7,numerica,manter,decisão humana baseada no conjunto de evidências,PSI T-V=0.014; SHAP=0.1113; VIF=1.16


,features
decisao,
manter,12
remover,2
manter_com_tratamento,1


**Decisão humana aprovada:** o conjunto final pré-OOT tem **13 features originais**. Removidas: **cat_var6, cat_var13**.

- `var12` será representada exclusivamente por `var12_estado`.
- `cat_var6` foi removida por PSI material combinado a contribuição incremental não demonstrada.
- `cat_var13` foi removida por robustez: há sinal, mas a instabilidade temporal é elevada em relação ao ganho incremental. Ela não é classificada como leakage.
- `cat_var10` permanece com monitoramento temporal obrigatório.
- `var1` e `var3` permanecem como redundância controlável; nenhuma feature derivada foi criada.

## 18. Candidato final pré-OOT

**Comentário Técnico:** a partir deste ponto existe **um único candidato**. As seções seguintes — estabilidade por safra e calibração — avaliam exatamente esse candidato e não uma variante. As alternativas testadas neste notebook permanecem registradas como análise de sensibilidade de desenvolvimento e **não** serão selecionadas com o OOT.

In [27]:
numericas_finais = [v for v in VARIAVEIS_NUMERICAS if v in features_finais]
categoricas_finais = [v for v in VARIAVEIS_CATEGORICAS if v in features_finais]
X_candidato_treino = preparar_var12_categorico(
    X_treino[features_finais], numericas_finais, categoricas_finais)
X_candidato_validacao = preparar_var12_categorico(
    X_validacao[features_finais], numericas_finais, categoricas_finais)
categoricas_candidato = categoricas_finais + ['var12_estado']
modelo_candidato = CatBoostClassifier(iterations=1500, depth=5, learning_rate=0.05,
    loss_function='Logloss', eval_metric='AUC', random_seed=42, verbose=False,
    allow_writing_files=False, thread_count=-1)
modelo_candidato.fit(X_candidato_treino, y_treino, cat_features=categoricas_candidato,
    eval_set=(X_candidato_validacao, y_validacao), early_stopping_rounds=100, verbose=False)
prob_candidato_treino = modelo_candidato.predict_proba(X_candidato_treino)[:, 1]
prob_candidato_validacao = modelo_candidato.predict_proba(X_candidato_validacao)[:, 1]
amostra_shap_final = X_candidato_validacao.sample(
    n=min(3000, len(X_candidato_validacao)), random_state=42)
valores_shap_final = modelo_candidato.get_feature_importance(
    Pool(amostra_shap_final, cat_features=categoricas_candidato), type='ShapValues')[:, :-1]
tabela_shap_final = pd.DataFrame({'feature': amostra_shap_final.columns,
    'shap_abs_medio': np.abs(valores_shap_final).mean(axis=0),
    'shap_medio': valores_shap_final.mean(axis=0)}).sort_values('shap_abs_medio', ascending=False)
assert features_finais == FEATURES_FINAIS_ORIGINAIS
assert features_removidas == FEATURES_REMOVIDAS
assert list(X_candidato_treino.columns) == VARIAVEIS_MODELO_FINAL
assert 'var12_estado' in X_candidato_treino and 'var12' not in X_candidato_treino
assert set(X_candidato_treino['var12_estado'].unique()) <= {'99997', '99998', '99999', 'REGULAR', 'MISSING'}
print('Candidato final humano aprovado: 13 features originais, var12_estado categórica.')

metricas_candidato = pd.DataFrame([
    {'amostra': 'Treino', **calcular_metricas_classificacao(y_treino, prob_candidato_treino)},
    {'amostra': 'Validação', **calcular_metricas_classificacao(y_validacao, prob_candidato_validacao)},
])
display(metricas_candidato.style.format(
    {c: '{:.4f}' for c in metricas_candidato.columns if c != 'amostra'}).hide(axis='index'))
assert set(X_candidato_treino.index).isdisjoint(indices_oot_congelado)
assert set(X_candidato_validacao.index).isdisjoint(indices_oot_congelado)
print(f'Candidato unico: CatBoost depth={configuracao_catboost[0]} '
      f'learning_rate={configuracao_catboost[1]:g} '
      f'best_iteration={modelo_candidato.get_best_iteration()} '
      f'features={len(features_finais)}')

Candidato final humano aprovado: 13 features originais, var12_estado categórica.


amostra,roc_auc,gini,ks,pr_auc,brier
Treino,0.8469,0.6938,0.5421,0.5540,0.0731
Validação,0.8421,0.6843,0.5301,0.5844,0.0760


Candidato unico: CatBoost depth=5 learning_rate=0.05 best_iteration=610 features=13


## 19. Estabilidade do candidato na Validação por safra

**Pergunta:** o desempenho do candidato se mantém quando a Validação é aberta por safra?

Esta é a primeira verificação de estabilidade temporal real do modelo. Ela usa somente 2019-09 e 2019-10. Com apenas duas safras, o resultado é indicativo: ele pode revelar uma ruptura, mas não pode comprovar estabilidade. A verificação completa depende do OOT.

In [28]:
serie_prob_candidato = pd.Series(prob_candidato_validacao, index=validacao.index)
registros_safra_validacao = []
for safra, grupo in validacao.groupby('safra'):
    alvo_safra = grupo[ALVO]
    probabilidade_safra = serie_prob_candidato.loc[grupo.index].to_numpy()
    registros_safra_validacao.append({
        'safra': safra.strftime('%Y-%m'), 'registros': len(grupo), 'eventos': int(alvo_safra.sum()),
        **calcular_metricas_classificacao(alvo_safra, probabilidade_safra),
        'taxa_evento': float(alvo_safra.mean()),
        'probabilidade_media': float(probabilidade_safra.mean()),
    })
registros_safra_validacao.append({
    'safra': 'Validação total', 'registros': len(validacao), 'eventos': int(y_validacao.sum()),
    **calcular_metricas_classificacao(y_validacao, prob_candidato_validacao),
    'taxa_evento': float(y_validacao.mean()),
    'probabilidade_media': float(prob_candidato_validacao.mean()),
})
tabela_safra_validacao = pd.DataFrame(registros_safra_validacao)
display(tabela_safra_validacao.style.format({
    **{c: '{:.4f}' for c in ['roc_auc', 'gini', 'ks', 'pr_auc', 'brier']},
    'taxa_evento': '{:.2%}', 'probabilidade_media': '{:.2%}'}).hide(axis='index'))
apenas_safras = tabela_safra_validacao[tabela_safra_validacao['safra'] != 'Validação total']
amplitude_auc_safra = float(apenas_safras['roc_auc'].max() - apenas_safras['roc_auc'].min())
amplitude_ks_safra = float(apenas_safras['ks'].max() - apenas_safras['ks'].min())
display(Markdown(
    f"**Análise/Interpretação:** entre as duas safras da Validação, o ROC-AUC varia "
    f"**{amplitude_auc_safra:.4f}** e o KS varia **{amplitude_ks_safra:.4f}**. A taxa observada do evento "
    f"vai de **{apenas_safras['taxa_evento'].min():.2%}** a **{apenas_safras['taxa_evento'].max():.2%}**, "
    f"enquanto a probabilidade média prevista vai de "
    f"**{apenas_safras['probabilidade_media'].min():.2%}** a "
    f"**{apenas_safras['probabilidade_media'].max():.2%}**. "
    f"Foi observada amplitude de aproximadamente **{amplitude_auc_safra:.3f}** de ROC-AUC entre as duas "
    "safras de Validação, indicando variabilidade temporal relevante antes da avaliação OOT. Duas safras "
    "não permitem definir normalidade nem concluir estabilidade temporal."
))

safra,registros,eventos,roc_auc,gini,ks,pr_auc,brier,taxa_evento,probabilidade_media
2019-09,16134,1993,0.8541,0.7083,0.5499,0.5957,0.0736,12.35%,13.78%
2019-10,16487,2100,0.8307,0.6614,0.5124,0.5739,0.0784,12.74%,13.13%
Validação total,32621,4093,0.8421,0.6843,0.5301,0.5844,0.0760,12.55%,13.46%


**Análise/Interpretação:** entre as duas safras da Validação, o ROC-AUC varia **0.0235** e o KS varia **0.0374**. A taxa observada do evento vai de **12.35%** a **12.74%**, enquanto a probabilidade média prevista vai de **13.13%** a **13.78%**. Foi observada amplitude de aproximadamente **0.023** de ROC-AUC entre as duas safras de Validação, indicando variabilidade temporal relevante antes da avaliação OOT. Duas safras não permitem definir normalidade nem concluir estabilidade temporal.

## 20. Calibração em desenvolvimento

**Pergunta:** as probabilidades do candidato estão próximas das taxas observadas, e em que escala?

**Comentário Técnico:** nenhuma regra automática de aprovação é aplicada. Os números são reportados em valor absoluto e também **relativos à taxa observada**, porque um viés de um ponto percentual tem significado muito diferente conforme a taxa base. Nenhum método de recalibração é implementado nesta etapa.

In [29]:
def erro_calibracao_esperado(alvo, probabilidade, quantidade_faixas=10):
    """Erro esperado de calibração com faixas de mesma quantidade de registros."""

    quadro = pd.DataFrame({'alvo': np.asarray(alvo), 'probabilidade': np.asarray(probabilidade)})
    quadro['faixa'] = pd.qcut(quadro['probabilidade'].rank(method='first'), quantidade_faixas, labels=False)
    resumo = quadro.groupby('faixa').agg(observado=('alvo', 'mean'),
        previsto=('probabilidade', 'mean'), quantidade=('alvo', 'size'))
    return float(((resumo['observado'] - resumo['previsto']).abs() * resumo['quantidade'] / len(quadro)).sum())

taxa_observada = float(y_validacao.mean())
probabilidade_media = float(prob_candidato_validacao.mean())
ece_candidato = erro_calibracao_esperado(y_validacao, prob_candidato_validacao)
vies_absoluto = probabilidade_media - taxa_observada
diagnostico_calibracao = pd.DataFrame([{
    'modelo': 'CatBoost candidato',
    'brier_validacao': calcular_metricas_classificacao(y_validacao, prob_candidato_validacao)['brier'],
    'ece_validacao': ece_candidato,
    'taxa_observada': taxa_observada,
    'probabilidade_media': probabilidade_media,
    'vies_absoluto_pp': vies_absoluto * 100,
    'vies_relativo': vies_absoluto / taxa_observada,
    'ece_relativo': ece_candidato / taxa_observada,
}])
display(diagnostico_calibracao.style.format({
    'brier_validacao': '{:.4f}', 'ece_validacao': '{:.4f}', 'taxa_observada': '{:.2%}',
    'probabilidade_media': '{:.2%}', 'vies_absoluto_pp': '{:+.2f}',
    'vies_relativo': '{:+.1%}', 'ece_relativo': '{:.1%}'}).hide(axis='index'))

observado_faixa, previsto_faixa = calibration_curve(y_validacao, prob_candidato_validacao,
                                                    n_bins=10, strategy='quantile')
fig_calibracao_candidato = go.Figure()
fig_calibracao_candidato.add_trace(go.Scatter(x=previsto_faixa, y=observado_faixa, mode='lines+markers',
    name='CatBoost candidato', line=dict(color=CORES['principal'], width=3)))
limite_grafico = float(max(previsto_faixa.max(), observado_faixa.max()))
fig_calibracao_candidato.add_trace(go.Scatter(x=[0, limite_grafico], y=[0, limite_grafico],
    mode='lines', name='Calibração perfeita', line=dict(color=CORES['cinza'], dash='dash')))
aplicar_layout_executivo(fig_calibracao_candidato, 'Calibração do candidato na Validação',
    titulo_eixo_x='Probabilidade média prevista', titulo_eixo_y='Taxa observada')
aplicar_eixo_percentual(fig_calibracao_candidato, 'x')
aplicar_eixo_percentual(fig_calibracao_candidato, 'y')
fig_calibracao_candidato.show()
salvar_grafico(fig_calibracao_candidato, 'final_11_calibracao_candidato', PASTA_FIGURAS)

decisao_calibracao = 'não recalibrar no desenvolvimento'
display(Markdown(
    f"**Análise/Interpretação:** o candidato prevê **{probabilidade_media:.2%}** contra "
    f"**{taxa_observada:.2%}** observados na Validação, um viés de **{vies_absoluto * 100:+.2f} p.p.**, "
    f"equivalente a **{vies_absoluto / taxa_observada:+.1%}** em termos relativos. O ECE é "
    f"**{ece_candidato:.4f}**, ou **{ece_candidato / taxa_observada:.1%}** da taxa observada, e o Brier é "
    f"**{diagnostico_calibracao.iloc[0]['brier_validacao']:.4f}**. Em termos absolutos os desvios são "
    "pequenos; em termos relativos existe sobre-previsão moderada, e a leitura relativa é a que importa "
    "quando a probabilidade prevista for usada como insumo de decisão."
))
display(Markdown(
    f"**Decisão de desenvolvimento: {decisao_calibracao}.** As ressalvas registradas são explícitas: "
    "(1) a Validação também participou do early stopping e do tuning, portanto as métricas de calibração "
    "medidas nela **não** são uma estimativa final independente e tendem ao otimismo; "
    "(2) a calibração definitiva será avaliada **somente no OOT**; "
    "(3) o OOT **não** será utilizado para ajustar posteriormente um calibrador — caso haja necessidade "
    "de recalibração, ela deverá ser ajustada em partição interna ao Treino e reavaliada em um ciclo "
    "seguinte, nunca sobre a amostra fora do tempo já observada."
))

modelo,brier_validacao,ece_validacao,taxa_observada,probabilidade_media,vies_absoluto_pp,vies_relativo,ece_relativo
CatBoost candidato,0.0760,0.0093,12.55%,13.46%,+0.91,+7.2%,7.4%


**Análise/Interpretação:** o candidato prevê **13.46%** contra **12.55%** observados na Validação, um viés de **+0.91 p.p.**, equivalente a **+7.2%** em termos relativos. O ECE é **0.0093**, ou **7.4%** da taxa observada, e o Brier é **0.0760**. Em termos absolutos os desvios são pequenos; em termos relativos existe sobre-previsão moderada, e a leitura relativa é a que importa quando a probabilidade prevista for usada como insumo de decisão.

**Decisão de desenvolvimento: não recalibrar no desenvolvimento.** As ressalvas registradas são explícitas: (1) a Validação também participou do early stopping e do tuning, portanto as métricas de calibração medidas nela **não** são uma estimativa final independente e tendem ao otimismo; (2) a calibração definitiva será avaliada **somente no OOT**; (3) o OOT **não** será utilizado para ajustar posteriormente um calibrador — caso haja necessidade de recalibração, ela deverá ser ajustada em partição interna ao Treino e reavaliada em um ciclo seguinte, nunca sobre a amostra fora do tempo já observada.

## 21. Configuração congelada para a avaliação OOT

Registro único da configuração que será levada ao OOT. Qualquer alteração posterior invalida o congelamento e exige nova aprovação humana.

In [30]:
variantes_sensibilidade = pd.DataFrame([
    {'variante': 'var12 híbrida: contínua + três indicadores',
     'delta_roc_auc_validacao': -comparacao_var12.iloc[1]['delta_roc_auc'],
     'motivo_do_registro': 'experimento anterior preservado; não é a representação congelada'},
    {'variante': 'sem cat_var10 e cat_var13',
     'delta_roc_auc_validacao': tabela_ablacao.set_index('cenario').loc[
         'Sem cat_var10 e cat_var13', 'delta_roc_auc_vs_completo'],
     'motivo_do_registro': 'remove as duas features de maior PSI'},
    {'variante': 'sem var1 e var3',
     'delta_roc_auc_validacao': tabela_ablacao.set_index('cenario').loc[
         'Sem var1 e var3', 'delta_roc_auc_vs_completo'],
     'motivo_do_registro': 'remove o par de maior correlação'},
])
variantes_sensibilidade['status'] = 'sensibilidade de desenvolvimento; não será selecionada com OOT'
configuracao_congelada = pd.DataFrame({
    'componente': ['modelo', 'features', 'quantidade_features', 'features_removidas', 'var12', 'cat_var13',
                   'cat_var10', 'cat_var6', 'var1 e var3', 'missing numérico', 'missing categórico',
                   'depth', 'learning_rate', 'iterations_max', 'early_stopping_rounds',
                   'best_iteration_desenvolvimento', 'calibração', 'seed', 'amostras de desenvolvimento',
                   'OOT'],
    'configuracao': ['CatBoost', ', '.join(features_finais), len(features_finais),
                     ', '.join(features_removidas) if features_removidas else 'nenhuma',
                     'var12_estado: 99997/99998/99999/REGULAR; MISSING separado se surgir',
                     f"{decisoes_por_variavel['cat_var13']}; sinal existente, porém instabilidade temporal "
                     'elevada em relação ao ganho incremental; não classificada como leakage',
                     f"{decisoes_por_variavel['cat_var10']}; PSI "
                     f"{tabela_features.set_index('variavel').loc['cat_var10', 'estabilidade_psi']:.3f}; "
                     'monitoramento obrigatório',
                     f"{decisoes_por_variavel['cat_var6']}; PSI "
                     f"{tabela_features.set_index('variavel').loc['cat_var6', 'estabilidade_psi']:.3f}; "
                     f"ablação ΔAUC="
                     f"{tabela_ablacao.set_index('cenario').loc['Sem cat_var6', 'delta_roc_auc_vs_completo']:+.4f} "
                     'não distinguível do ruído',
                     f'mantidas; diagnóstico do par: {veredito_par}',
                     'nativo do CatBoost', '__MISSING__',
                     configuracao_catboost[0], configuracao_catboost[1], 1500, 100,
                     modelo_candidato.get_best_iteration(), decisao_calibracao, SEMENTE_ALEATORIA,
                     'Treino 2019-01 a 2019-08; Validação 2019-09 a 2019-10',
                     '2019-11 a 2020-01; congelado e não utilizado até o gate pré-OOT']})
display(configuracao_congelada.style.hide(axis='index'))
display(Markdown('**Variantes registradas apenas como sensibilidade de desenvolvimento**'))
display(variantes_sensibilidade.style.format({'delta_roc_auc_validacao': '{:+.4f}'}).hide(axis='index'))
display(Markdown(
    '**Champion de desenvolvimento recomendado: CatBoost com o conjunto final de '
    f'{len(features_finais)} features.** A configuração acima é a **candidata única** à avaliação OOT e '
    '**não é champion final**. As variantes existem para documentar sensibilidade e não entram em '
    'nenhuma comparação que utilize o OOT.'
))

componente,configuracao
modelo,CatBoost
features,"var1, var3, var4, var5, var7, var8, var9, var11, var12, var14, cat_var2, cat_var10, cat_var15"
quantidade_features,13
features_removidas,"cat_var6, cat_var13"
var12,var12_estado: 99997/99998/99999/REGULAR; MISSING separado se surgir
cat_var13,"remover; sinal existente, porém instabilidade temporal elevada em relação ao ganho incremental; não classificada como leakage"
cat_var10,manter; PSI 0.165; monitoramento obrigatório
cat_var6,remover; PSI 0.110; ablação ΔAUC=-0.0004 não distinguível do ruído
var1 e var3,mantidas; diagnóstico do par: redundância controlável
missing numérico,nativo do CatBoost


**Variantes registradas apenas como sensibilidade de desenvolvimento**

variante,delta_roc_auc_validacao,motivo_do_registro,status
var12 híbrida: contínua + três indicadores,+0.0003,experimento anterior preservado; não é a representação congelada,sensibilidade de desenvolvimento; não será selecionada com OOT
sem cat_var10 e cat_var13,-0.0031,remove as duas features de maior PSI,sensibilidade de desenvolvimento; não será selecionada com OOT
sem var1 e var3,-0.0488,remove o par de maior correlação,sensibilidade de desenvolvimento; não será selecionada com OOT


**Champion de desenvolvimento recomendado: CatBoost com o conjunto final de 13 features.** A configuração acima é a **candidata única** à avaliação OOT e **não é champion final**. As variantes existem para documentar sensibilidade e não entram em nenhuma comparação que utilize o OOT.

## Gate antes da avaliação final

Até este ponto do fluxo não haviam sido calculados AUC, Gini, KS, PR-AUC, Brier, SHAP ou lift preditivo no OOT. Ablação, calibração e explicabilidade haviam sido avaliadas somente em desenvolvimento. A fase OOT começa na seção 22, depois de o protocolo D019 ter sido definido e antes da primeira predição OOT nesta execução. A transformação do score é tratada somente após a avaliação final; faixas de risco e recomendações operacionais permanecem fora do escopo.

Foram fechados neste gate: redundância de `var1` e `var3` com VIF, estabilidade de sinal e ablação; teste de `cat_var13` condicional à safra; ablação de `cat_var6`; caracterização de `var12` e comparação de representações; estabilidade do candidato por safra da Validação; narrativa de calibração em termos absolutos e relativos; e decisão final de features. **Nenhuma feature permanece com decisão `avaliar`.**

As decisões humanas do conjunto final, representação de `var12`, calibração e configuração foram incorporadas e congeladas neste gate histórico. A autorização e o protocolo para a avaliação OOT estão documentados na decisão D019 antes de qualquer predição fora do tempo.

In [31]:
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
tabelas_finais = {
    'final_resumo_dados.csv': resumo_dados,
    'final_split.csv': tabela_split,
    'final_feature_analysis.csv': tabela_features,
    'final_vif_numericas.csv': tabela_vif,
    'final_var12.csv': tabela_var12_dev,
    'final_var12_caracterizacao.csv': caracterizacao_var12,
    'final_var12_representacoes.csv': comparacao_var12,
    'final_cat_var13_por_safra.csv': tabela_cat13_safra,
    'final_cat_var13_resumo.csv': resumo_cat13,
    'final_model_metrics.csv': tabela_metricas,
    'final_lift_validation.csv': tabela_lift,
    'final_tuning_logistica.csv': tabela_tuning_logistica,
    'final_tuning_catboost.csv': tabela_tuning_catboost,
    'final_coeficientes_var1_var3_por_c.csv': tabela_coeficientes_c,
    'final_ablation_features.csv': tabela_ablacao,
    'final_par_var1_var3.csv': tabela_par_var1_var3,
    'final_logistic_coefficients.csv': coeficientes_logistica,
    'final_shap_importance.csv': tabela_shap_final,
    'final_feature_decision.csv': tabela_decisao_features,
    'final_metricas_candidato.csv': metricas_candidato,
    'final_validacao_por_safra.csv': tabela_safra_validacao,
    'final_calibration.csv': diagnostico_calibracao,
    'final_variantes_sensibilidade.csv': variantes_sensibilidade,
    'final_frozen_candidate.csv': configuracao_congelada,
}
for nome, tabela in tabelas_finais.items():
    tabela.to_csv(PASTA_TABELAS / nome, index=False, encoding='utf-8-sig')

# Verificacoes finais de congelamento do OOT.
assert base_original.equals(base.drop(columns=['safra', 'amostra']))
assert not any('oot' in chave[1].lower() for chave in probabilidades)
indices_de_modelagem = (set(X_treino.index) | set(X_validacao.index)
                        | set(X_cat_treino.index) | set(X_cat_validacao.index)
                        | set(X_var12cat_treino.index) | set(X_var12cat_validacao.index)
                        | set(X_candidato_treino.index) | set(X_candidato_validacao.index)
                        | set(amostra_shap.index) | set(amostra_shap_final.index))
assert indices_de_modelagem.isdisjoint(indices_oot_congelado)
assert features_finais == FEATURES_FINAIS_ORIGINAIS
assert features_removidas == FEATURES_REMOVIDAS
assert 'avaliar' not in set(tabela_decisao_features['decisao_final'])
assert 'var12_estado' in X_candidato_treino.columns and 'var12' not in X_candidato_treino.columns
assert 'cat_var6' not in X_candidato_treino.columns and 'cat_var13' not in X_candidato_treino.columns
safras_utilizadas = set(base.loc[sorted(indices_de_modelagem), 'safra'].dt.strftime('%Y-%m'))
assert safras_utilizadas.isdisjoint({'2019-11', '2019-12', '2020-01'})
print(f'{len(tabelas_finais)} tabelas finais exportadas.')
print(f'Safras utilizadas em modelagem: {sorted(safras_utilizadas)}')
print(f'Até o gate pré-OOT: {len(indices_oot_congelado):,} registros sem pontuação nem uso em fit/seleção.')
print(f'Conjunto final: {len(features_finais)} features | removidas: '
      f'{features_removidas if features_removidas else "nenhuma"}')

24 tabelas finais exportadas.
Safras utilizadas em modelagem: ['2019-01', '2019-02', '2019-03', '2019-04', '2019-05', '2019-06', '2019-07', '2019-08', '2019-09', '2019-10']
Até o gate pré-OOT: 53,098 registros sem pontuação nem uso em fit/seleção.
Conjunto final: 13 features | removidas: ['cat_var6', 'cat_var13']


## 22. Avaliação final fora do tempo — OOT

Esta seção inicia a primeira avaliação independente de desempenho preditivo do candidato congelado. O checkpoint pré-OOT é o commit `0c5c6a7`, e o protocolo **D019 foi definido antes da primeira predição OOT nesta execução**.

**Comentário Técnico:** o OOT já havia sido caracterizado descritivamente antes do congelamento, mas não participou de fit, tuning, seleção de features, escolha de hiperparâmetros, calibração, SHAP ou avaliação preditiva. A partir deste ponto ele é utilizado exclusivamente para avaliação final; nenhum resultado poderá alterar o champion, as features, os tratamentos ou a calibração.

### 22.1 Refit final e preflight

O candidato de desenvolvimento usa Treino no fit e Validação para early stopping. O modelo final usa Treino + Validação no fit, quantidade fixa de árvores lida de `modelo_candidato.tree_count_`, nenhum early stopping e nenhuma exposição do OOT durante o ajuste.

A Regressão Logística é reconstruída com a especificação final de 13 features e `C = 0.1`, somente como benchmark.

In [32]:
features_originais_congeladas = FEATURES_FINAIS_ORIGINAIS.copy()
numericas_finais_congeladas = VARIAVEIS_NUMERICAS_FINAIS.copy()
categoricas_finais_congeladas = VARIAVEIS_CATEGORICAS_FINAIS.copy()
colunas_efetivas_congeladas = VARIAVEIS_MODELO_FINAL.copy()
safras_desenvolvimento_esperadas = {
    '2019-01', '2019-02', '2019-03', '2019-04', '2019-05',
    '2019-06', '2019-07', '2019-08', '2019-09', '2019-10',
}
safras_oot_esperadas = {'2019-11', '2019-12', '2020-01'}

# O número efetivo vem do objeto congelado; não é inferido de best_iteration.
quantidade_arvores_congelada = int(modelo_candidato.tree_count_)
best_iteration_congelado = int(modelo_candidato.get_best_iteration())

desenvolvimento = pd.concat([treino, validacao], axis=0).sort_index()
oot = base.loc[base['amostra'].eq('OOT')].copy()
X_desenvolvimento_original = desenvolvimento[features_originais_congeladas]
y_desenvolvimento = desenvolvimento[ALVO]
X_oot_original = oot[features_originais_congeladas]
y_oot = oot[ALVO]

X_candidato_desenvolvimento = preparar_var12_categorico(
    X_desenvolvimento_original,
    [v for v in features_originais_congeladas if v in VARIAVEIS_NUMERICAS],
    [v for v in features_originais_congeladas if v in VARIAVEIS_CATEGORICAS],
)
X_candidato_oot = preparar_var12_categorico(
    X_oot_original,
    [v for v in features_originais_congeladas if v in VARIAVEIS_NUMERICAS],
    [v for v in features_originais_congeladas if v in VARIAVEIS_CATEGORICAS],
)
categoricas_candidato_final = categoricas_finais_congeladas + ['var12_estado']

modelo_catboost_final = CatBoostClassifier(
    iterations=quantidade_arvores_congelada,
    depth=5,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1,
)
indices_fit_catboost_final = set(X_candidato_desenvolvimento.index)
modelo_catboost_final.fit(
    X_candidato_desenvolvimento,
    y_desenvolvimento,
    cat_features=categoricas_candidato_final,
    verbose=False,
)

def construir_pipeline_logistica_final() -> Pipeline:
    """Constrói o benchmark congelado com a representação final de features."""

    preprocessamento = ColumnTransformer([
        ('numericas', Pipeline([
            ('imputador', SimpleImputer(strategy='median', add_indicator=True)),
            ('escala', StandardScaler()),
        ]), numericas_finais_congeladas),
        ('categoricas', Pipeline([
            ('texto', ConversorCategoricoTexto()),
            ('one_hot', OneHotEncoder(handle_unknown='ignore')),
        ]), categoricas_candidato_final),
    ])
    return Pipeline([
        ('preprocessamento', preprocessamento),
        ('modelo', LogisticRegression(
            C=0.1,
            l1_ratio=0.0,
            max_iter=2000,
            random_state=42,
        )),
    ])

# Referência de Validação da especificação final, sem tuning adicional e sem OOT.
pipeline_logistica_validacao_final = construir_pipeline_logistica_final()
pipeline_logistica_validacao_final.fit(X_candidato_treino, y_treino)
prob_logistica_validacao_final = pipeline_logistica_validacao_final.predict_proba(
    X_candidato_validacao
)[:, 1]

pipeline_logistica_final = construir_pipeline_logistica_final()
indices_fit_logistica_final = set(X_candidato_desenvolvimento.index)
pipeline_logistica_final.fit(X_candidato_desenvolvimento, y_desenvolvimento)

calibrador_final = None
parametros_catboost_final = modelo_catboost_final.get_params()
indices_desenvolvimento = set(desenvolvimento.index)
indices_oot = set(oot.index)
estados_var12_observados = set(
    pd.concat([
        X_candidato_desenvolvimento['var12_estado'],
        X_candidato_oot['var12_estado'],
    ]).unique()
)

# Assertions do protocolo antes da primeira predição OOT.
assert features_finais == features_originais_congeladas
assert features_removidas == FEATURES_REMOVIDAS
assert list(X_candidato_desenvolvimento.columns) == colunas_efetivas_congeladas
assert list(X_candidato_oot.columns) == colunas_efetivas_congeladas
assert 'cat_var6' not in X_candidato_desenvolvimento
assert 'cat_var13' not in X_candidato_desenvolvimento
assert 'var12_estado' in X_candidato_desenvolvimento
assert 'var12' not in X_candidato_desenvolvimento
assert estados_var12_observados <= {'99997', '99998', '99999', 'REGULAR', 'MISSING'}
assert best_iteration_congelado == 610
assert quantidade_arvores_congelada == int(modelo_candidato.tree_count_)
assert int(modelo_catboost_final.tree_count_) == quantidade_arvores_congelada
assert parametros_catboost_final['iterations'] == quantidade_arvores_congelada
assert parametros_catboost_final['depth'] == 5
assert parametros_catboost_final['learning_rate'] == 0.05
assert parametros_catboost_final['loss_function'] == 'Logloss'
assert parametros_catboost_final['eval_metric'] == 'AUC'
assert parametros_catboost_final['random_seed'] == 42
assert modelo_catboost_final.get_best_iteration() in (None, -1)
assert calibrador_final is None
assert pipeline_logistica_final.get_params()['modelo__C'] == 0.1
assert pipeline_logistica_final.get_params()['modelo__l1_ratio'] == 0.0
assert indices_desenvolvimento.isdisjoint(indices_oot)
assert indices_fit_catboost_final == indices_desenvolvimento
assert indices_fit_logistica_final == indices_desenvolvimento
assert indices_fit_catboost_final.isdisjoint(indices_oot)
assert indices_fit_logistica_final.isdisjoint(indices_oot)
assert set(desenvolvimento['safra'].dt.strftime('%Y-%m')) == safras_desenvolvimento_esperadas
assert set(oot['safra'].dt.strftime('%Y-%m')) == safras_oot_esperadas
assert len(desenvolvimento) == len(treino) + len(validacao)
assert len(oot) == len(indices_oot_congelado)
assert base_original.equals(base.drop(columns=['safra', 'amostra']))

resumo_protocolo_oot = pd.DataFrame([
    {'item': 'checkpoint pré-OOT', 'valor': '0c5c6a7'},
    {'item': 'best_iteration de desenvolvimento', 'valor': best_iteration_congelado},
    {'item': 'tree_count_ congelado', 'valor': quantidade_arvores_congelada},
    {'item': 'fit CatBoost final', 'valor': 'Treino + Validação'},
    {'item': 'early stopping no refit', 'valor': 'não'},
    {'item': 'OOT no fit', 'valor': 'zero linhas'},
    {'item': 'calibração', 'valor': 'nenhuma'},
    {'item': 'features originais', 'valor': len(features_originais_congeladas)},
])
display(resumo_protocolo_oot.style.hide(axis='index'))
print('PROTOCOLO OOT VALIDADO — MODELO CONGELADO')

item,valor
checkpoint pré-OOT,0c5c6a7
best_iteration de desenvolvimento,610
tree_count_ congelado,611
fit CatBoost final,Treino + Validação
early stopping no refit,não
OOT no fit,zero linhas
calibração,nenhuma
features originais,13


PROTOCOLO OOT VALIDADO — MODELO CONGELADO


### 22.2 Abertura do OOT e métricas agregadas

Somente após o preflight acima são geradas as probabilidades OOT. As métricas de Validação são de desenvolvimento e carregam algum otimismo, pois essa amostra participou de early stopping, tuning e decisões de features. O OOT é a primeira estimativa independente de generalização do modelo congelado.

In [33]:
# Primeiras predições OOT desta fase, executadas somente após o preflight.
prob_catboost_desenvolvimento = modelo_catboost_final.predict_proba(
    X_candidato_desenvolvimento
)[:, 1]
prob_catboost_oot = modelo_catboost_final.predict_proba(X_candidato_oot)[:, 1]
prob_logistica_desenvolvimento = pipeline_logistica_final.predict_proba(
    X_candidato_desenvolvimento
)[:, 1]
prob_logistica_oot = pipeline_logistica_final.predict_proba(X_candidato_oot)[:, 1]

def resumir_metricas_oot(alvo, probabilidade) -> dict[str, float]:
    """Resume discriminação e calibração com as definições fixadas em D019."""

    metricas = calcular_metricas_classificacao(alvo, probabilidade)
    taxa = float(np.mean(alvo))
    media = float(np.mean(probabilidade))
    vies = media - taxa
    return {
        **metricas,
        'ece': erro_calibracao_esperado(alvo, probabilidade, quantidade_faixas=10),
        'taxa_observada': taxa,
        'probabilidade_media': media,
        'vies_absoluto': vies,
        'vies_relativo': vies / taxa,
    }

diagnosticos_modelos_oot = {
    ('CatBoost', 'Validação de desenvolvimento'): resumir_metricas_oot(
        y_validacao, prob_candidato_validacao
    ),
    ('CatBoost', 'OOT'): resumir_metricas_oot(y_oot, prob_catboost_oot),
    ('Regressão Logística', 'Validação de desenvolvimento'): resumir_metricas_oot(
        y_validacao, prob_logistica_validacao_final
    ),
    ('Regressão Logística', 'OOT'): resumir_metricas_oot(y_oot, prob_logistica_oot),
}

metricas_comparacao = ['roc_auc', 'gini', 'ks', 'pr_auc', 'brier', 'ece']
rotulos_metricas = {
    'roc_auc': 'ROC-AUC',
    'gini': 'Gini',
    'ks': 'KS',
    'pr_auc': 'PR-AUC (Average Precision)',
    'brier': 'Brier',
    'ece': 'ECE',
}
registros_comparacao_oot = []
for nome_modelo in ['CatBoost', 'Regressão Logística']:
    validacao_modelo = diagnosticos_modelos_oot[
        (nome_modelo, 'Validação de desenvolvimento')
    ]
    oot_modelo = diagnosticos_modelos_oot[(nome_modelo, 'OOT')]
    for metrica in metricas_comparacao:
        registros_comparacao_oot.append({
            'modelo': nome_modelo,
            'metrica': rotulos_metricas[metrica],
            'validacao_desenvolvimento': validacao_modelo[metrica],
            'oot': oot_modelo[metrica],
            'delta_validacao_oot': oot_modelo[metrica] - validacao_modelo[metrica],
        })
tabela_comparacao_oot = pd.DataFrame(registros_comparacao_oot)

registros_calibracao_agregada = []
for (nome_modelo, amostra), diagnostico in diagnosticos_modelos_oot.items():
    registros_calibracao_agregada.append({
        'modelo': nome_modelo,
        'amostra': amostra,
        **{chave: diagnostico[chave] for chave in [
            'brier', 'ece', 'taxa_observada', 'probabilidade_media',
            'vies_absoluto', 'vies_relativo',
        ]},
    })
tabela_calibracao_agregada_oot = pd.DataFrame(registros_calibracao_agregada)

display(tabela_comparacao_oot.style.format({
    'validacao_desenvolvimento': '{:.4f}',
    'oot': '{:.4f}',
    'delta_validacao_oot': '{:+.4f}',
}).hide(axis='index'))
display(tabela_calibracao_agregada_oot.style.format({
    'brier': '{:.4f}',
    'ece': '{:.4f}',
    'taxa_observada': '{:.2%}',
    'probabilidade_media': '{:.2%}',
    'vies_absoluto': '{:+.2%}',
    'vies_relativo': '{:+.1%}',
}).hide(axis='index'))
print(f'Predições OOT geradas: CatBoost={len(prob_catboost_oot):,}; '
      f'Logística={len(prob_logistica_oot):,}. Nenhum ajuste posterior foi executado.')

modelo,metrica,validacao_desenvolvimento,oot,delta_validacao_oot
CatBoost,ROC-AUC,0.8421,0.8227,-0.0194
CatBoost,Gini,0.6843,0.6455,-0.0388
CatBoost,KS,0.5301,0.5027,-0.0275
CatBoost,PR-AUC (Average Precision),0.5844,0.4987,-0.0857
CatBoost,Brier,0.0760,0.0979,+0.0219
CatBoost,ECE,0.0093,0.0200,+0.0107
Regressão Logística,ROC-AUC,0.8188,0.8036,-0.0152
Regressão Logística,Gini,0.6377,0.6073,-0.0304
Regressão Logística,KS,0.4917,0.4774,-0.0143
Regressão Logística,PR-AUC (Average Precision),0.5353,0.4792,-0.0560


modelo,amostra,brier,ece,taxa_observada,probabilidade_media,vies_absoluto,vies_relativo
CatBoost,Validação de desenvolvimento,0.0760,0.0093,12.55%,13.46%,+0.91%,+7.2%
CatBoost,OOT,0.0979,0.0200,14.67%,15.44%,+0.78%,+5.3%
Regressão Logística,Validação de desenvolvimento,0.0807,0.0096,12.55%,13.39%,+0.84%,+6.7%
Regressão Logística,OOT,0.1008,0.0214,14.67%,15.24%,+0.58%,+3.9%


Predições OOT geradas: CatBoost=53,098; Logística=53,098. Nenhum ajuste posterior foi executado.


### 22.3 Intervalos de confiança OOT

Os intervalos abaixo quantificam a incerteza do CatBoost no OOT. São usados 500 bootstraps estratificados pelo target, seed 42 e intervalo percentil de 95%. Eles não orientam tuning nem alteração do modelo.

In [34]:
def bootstrap_estratificado_metricas(
    alvo,
    probabilidade,
    reamostragens: int = 500,
    semente: int = 42,
) -> pd.DataFrame:
    """Bootstrap estratificado para ROC-AUC, Gini, KS e Brier."""

    y = np.asarray(alvo, dtype=int)
    p = np.asarray(probabilidade, dtype=float)
    indices_zero = np.flatnonzero(y == 0)
    indices_um = np.flatnonzero(y == 1)
    if not len(indices_zero) or not len(indices_um):
        raise ValueError('Bootstrap exige as duas classes no OOT.')
    gerador = np.random.default_rng(semente)
    registros = []
    for _ in range(reamostragens):
        amostra_indices = np.concatenate([
            gerador.choice(indices_zero, size=len(indices_zero), replace=True),
            gerador.choice(indices_um, size=len(indices_um), replace=True),
        ])
        y_boot = y[amostra_indices]
        p_boot = p[amostra_indices]
        auc = float(roc_auc_score(y_boot, p_boot))
        fpr_boot, tpr_boot, _ = roc_curve(y_boot, p_boot)
        registros.append({
            'roc_auc': auc,
            'gini': 2 * auc - 1,
            'ks': float(np.max(np.abs(tpr_boot - fpr_boot))),
            'brier': float(np.mean((y_boot - p_boot) ** 2)),
        })
    return pd.DataFrame(registros)

amostras_bootstrap_oot = bootstrap_estratificado_metricas(
    y_oot, prob_catboost_oot, reamostragens=500, semente=42
)
estimativas_catboost_oot = diagnosticos_modelos_oot[('CatBoost', 'OOT')]
registros_ic_oot = []
for metrica in ['roc_auc', 'gini', 'ks', 'brier']:
    registros_ic_oot.append({
        'metrica': rotulos_metricas[metrica],
        'estimativa_oot': estimativas_catboost_oot[metrica],
        'ic95_inferior': float(amostras_bootstrap_oot[metrica].quantile(0.025)),
        'ic95_superior': float(amostras_bootstrap_oot[metrica].quantile(0.975)),
        'reamostragens': len(amostras_bootstrap_oot),
        'metodo': 'bootstrap estratificado percentil',
        'seed': 42,
    })
tabela_ic_catboost_oot = pd.DataFrame(registros_ic_oot)
display(tabela_ic_catboost_oot.style.format({
    'estimativa_oot': '{:.4f}',
    'ic95_inferior': '{:.4f}',
    'ic95_superior': '{:.4f}',
}).hide(axis='index'))
assert len(amostras_bootstrap_oot) == 500

metrica,estimativa_oot,ic95_inferior,ic95_superior,reamostragens,metodo,seed
ROC-AUC,0.8227,0.8171,0.8278,500,bootstrap estratificado percentil,42
Gini,0.6455,0.6342,0.6556,500,bootstrap estratificado percentil,42
KS,0.5027,0.4934,0.5149,500,bootstrap estratificado percentil,42
Brier,0.0979,0.0965,0.0994,500,bootstrap estratificado percentil,42


### 22.4 Performance por safra e variabilidade temporal observada

A tabela OOT separa 2019-11, 2019-12 e 2020-01. A visão temporal acrescenta 2019-09 e 2019-10 como referência de desenvolvimento. As origens têm escopos de ajuste distintos: a Validação foi pontuada pelo candidato treinado no Treino; o OOT foi pontuado pelo modelo final refitado em Treino + Validação. A diferença não compara o mesmo ajuste em duas amostras e não recebe interpretação causal.

In [35]:
def resumir_safra_modelo(
    grupo: pd.DataFrame,
    probabilidade: np.ndarray,
    origem: str,
) -> dict:
    """Calcula métricas mensais e preserva calibração se faltar uma classe."""

    alvo = grupo[ALVO].to_numpy()
    probabilidade = np.asarray(probabilidade, dtype=float)
    if len(np.unique(alvo)) == 2:
        metricas = calcular_metricas_classificacao(alvo, probabilidade)
    else:
        metricas = {
            'roc_auc': np.nan,
            'gini': np.nan,
            'ks': np.nan,
            'pr_auc': np.nan,
            'brier': float(np.mean((alvo - probabilidade) ** 2)),
        }
    return {
        'origem': origem,
        'safra': grupo['safra'].iloc[0].strftime('%Y-%m'),
        'registros': len(grupo),
        'eventos': int(alvo.sum()),
        'taxa_evento': float(alvo.mean()),
        'probabilidade_media': float(probabilidade.mean()),
        **metricas,
        'ece': erro_calibracao_esperado(alvo, probabilidade, quantidade_faixas=10),
    }

serie_prob_validacao_catboost = pd.Series(
    prob_candidato_validacao, index=validacao.index
)
serie_prob_oot_catboost = pd.Series(prob_catboost_oot, index=oot.index)

registros_validacao_temporal = []
for _, grupo in validacao.groupby('safra', sort=True):
    registros_validacao_temporal.append(resumir_safra_modelo(
        grupo,
        serie_prob_validacao_catboost.loc[grupo.index].to_numpy(),
        'Validação de desenvolvimento',
    ))

registros_oot_temporal = []
for _, grupo in oot.groupby('safra', sort=True):
    registros_oot_temporal.append(resumir_safra_modelo(
        grupo,
        serie_prob_oot_catboost.loc[grupo.index].to_numpy(),
        'OOT',
    ))

tabela_performance_oot_por_safra = pd.DataFrame(registros_oot_temporal)
tabela_performance_temporal = pd.DataFrame(
    registros_validacao_temporal + registros_oot_temporal
)
display(tabela_performance_oot_por_safra.style.format({
    'taxa_evento': '{:.2%}',
    'probabilidade_media': '{:.2%}',
    **{c: '{:.4f}' for c in ['roc_auc', 'gini', 'ks', 'pr_auc', 'brier', 'ece']},
}).hide(axis='index'))
display(Markdown('**Sequência temporal: Validação de desenvolvimento e OOT**'))
display(tabela_performance_temporal.style.format({
    'taxa_evento': '{:.2%}',
    'probabilidade_media': '{:.2%}',
    **{c: '{:.4f}' for c in ['roc_auc', 'gini', 'ks', 'pr_auc', 'brier', 'ece']},
}).hide(axis='index'))

fig_performance_oot = go.Figure()
fig_performance_oot.add_trace(go.Bar(
    x=tabela_performance_oot_por_safra['safra'],
    y=tabela_performance_oot_por_safra['roc_auc'],
    name='ROC-AUC',
    marker_color=CORES['principal'],
))
fig_performance_oot.add_trace(go.Bar(
    x=tabela_performance_oot_por_safra['safra'],
    y=tabela_performance_oot_por_safra['ks'],
    name='KS',
    marker_color=CORES['secundaria'],
))
aplicar_layout_executivo(
    fig_performance_oot,
    'ROC-AUC e KS por safra OOT',
    subtitulo='Três períodos discretos; barras agrupadas evitam sugerir tendência contínua',
    titulo_eixo_x='Safra',
    titulo_eixo_y='Métrica',
)
fig_performance_oot.update_layout(
    barmode='group',
    title_x=0.02,
    margin=dict(l=70, r=40, t=150, b=65),
    legend=dict(orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0.02),
)
fig_performance_oot.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=tabela_performance_oot_por_safra['safra'].tolist(),
)
fig_performance_oot.update_yaxes(range=[0, 1])
fig_performance_oot.show()
salvar_grafico(
    fig_performance_oot,
    'final_12_performance_oot_por_safra',
    PASTA_FIGURAS,
)

linha_auc_cat = tabela_comparacao_oot.query(
    "modelo == 'CatBoost' and metrica == 'ROC-AUC'"
).iloc[0]
linha_ks_cat = tabela_comparacao_oot.query(
    "modelo == 'CatBoost' and metrica == 'KS'"
).iloc[0]
ic_auc = tabela_ic_catboost_oot.query("metrica == 'ROC-AUC'").iloc[0]
display(Markdown(
    f"**FATO OBSERVADO:** a referência de desenvolvimento tem ROC-AUC "
    f"**{linha_auc_cat['validacao_desenvolvimento']:.4f}** no candidato treinado no Treino e "
    f"avaliado na Validação; a avaliação OOT tem ROC-AUC **{linha_auc_cat['oot']:.4f}** no modelo "
    f"final refitado em Treino + Validação. A diferença OOT menos referência é "
    f"**{linha_auc_cat['delta_validacao_oot']:+.4f}**; os valores vêm de ajustes distintos. "
    f"A diferença correspondente em KS é **{linha_ks_cat['delta_validacao_oot']:+.4f}**. "
    f"No OOT mensal, o ROC-AUC varia entre "
    f"**{tabela_performance_oot_por_safra['roc_auc'].min():.4f}** e "
    f"**{tabela_performance_oot_por_safra['roc_auc'].max():.4f}**; o IC95% do ROC-AUC OOT "
    f"agregado é **[{ic_auc['ic95_inferior']:.4f}; {ic_auc['ic95_superior']:.4f}]**.\n\n"
    "**HIPÓTESES:** mudanças de composição populacional e da taxa do evento podem coexistir com a "
    "variação observada, mas esta análise não identifica causas. Nenhuma hipótese altera o modelo."
))

origem,safra,registros,eventos,taxa_evento,probabilidade_media,roc_auc,gini,ks,pr_auc,brier,ece
OOT,2019-11,17344,2301,13.27%,14.19%,0.8209,0.6417,0.4960,0.4788,0.0905,0.0160
OOT,2019-12,17436,2285,13.11%,15.24%,0.7758,0.5516,0.4326,0.3293,0.1104,0.0374
OOT,2020-01,18318,3201,17.47%,16.83%,0.8588,0.7177,0.5770,0.6608,0.0929,0.0124


**Sequência temporal: Validação de desenvolvimento e OOT**

origem,safra,registros,eventos,taxa_evento,probabilidade_media,roc_auc,gini,ks,pr_auc,brier,ece
Validação de desenvolvimento,2019-09,16134,1993,12.35%,13.78%,0.8541,0.7083,0.5499,0.5957,0.0736,0.0143
Validação de desenvolvimento,2019-10,16487,2100,12.74%,13.13%,0.8307,0.6614,0.5124,0.5739,0.0784,0.0080
OOT,2019-11,17344,2301,13.27%,14.19%,0.8209,0.6417,0.4960,0.4788,0.0905,0.0160
OOT,2019-12,17436,2285,13.11%,15.24%,0.7758,0.5516,0.4326,0.3293,0.1104,0.0374
OOT,2020-01,18318,3201,17.47%,16.83%,0.8588,0.7177,0.5770,0.6608,0.0929,0.0124


**FATO OBSERVADO:** a referência de desenvolvimento tem ROC-AUC **0.8421** no candidato treinado no Treino e avaliado na Validação; a avaliação OOT tem ROC-AUC **0.8227** no modelo final refitado em Treino + Validação. A diferença OOT menos referência é **-0.0194**; os valores vêm de ajustes distintos. A diferença correspondente em KS é **-0.0275**. No OOT mensal, o ROC-AUC varia entre **0.7758** e **0.8588**; o IC95% do ROC-AUC OOT agregado é **[0.8171; 0.8278]**.

**HIPÓTESES:** mudanças de composição populacional e da taxa do evento podem coexistir com a variação observada, mas esta análise não identifica causas. Nenhuma hipótese altera o modelo.

### 22.5 Calibração OOT

A calibração é avaliada por curva, Brier, ECE, taxa observada, probabilidade média e vieses definidos no protocolo D019. Nenhum calibrador é ajustado.

In [36]:
diagnostico_calibracao_catboost_oot = tabela_calibracao_agregada_oot.query(
    "modelo == 'CatBoost' and amostra == 'OOT'"
).reset_index(drop=True)
display(diagnostico_calibracao_catboost_oot.style.format({
    'brier': '{:.4f}',
    'ece': '{:.4f}',
    'taxa_observada': '{:.2%}',
    'probabilidade_media': '{:.2%}',
    'vies_absoluto': '{:+.2%}',
    'vies_relativo': '{:+.1%}',
}).hide(axis='index'))

observado_oot_cal, previsto_oot_cal = calibration_curve(
    y_oot,
    prob_catboost_oot,
    n_bins=10,
    strategy='quantile',
)
fig_calibracao_oot = go.Figure()
fig_calibracao_oot.add_trace(go.Scatter(
    x=previsto_oot_cal,
    y=observado_oot_cal,
    mode='lines+markers',
    name='CatBoost final',
    line=dict(color=CORES['principal'], width=3),
))
limite_calibracao_oot = float(max(
    previsto_oot_cal.max(), observado_oot_cal.max()
) * 1.05)
fig_calibracao_oot.add_trace(go.Scatter(
    x=[0, limite_calibracao_oot],
    y=[0, limite_calibracao_oot],
    mode='lines',
    name='Calibração perfeita',
    line=dict(color=CORES['cinza'], dash='dash'),
))
aplicar_layout_executivo(
    fig_calibracao_oot,
    'Calibração do CatBoost no OOT',
    subtitulo='10 faixas equipopulacionais; nenhuma recalibração',
    titulo_eixo_x='Probabilidade média prevista',
    titulo_eixo_y='Taxa observada',
)
aplicar_eixo_percentual(fig_calibracao_oot, 'x')
aplicar_eixo_percentual(fig_calibracao_oot, 'y')
fig_calibracao_oot.update_layout(
    title_x=0.02,
    margin=dict(l=80, r=40, t=150, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0.02),
)
fig_calibracao_oot.update_xaxes(range=[0, limite_calibracao_oot])
fig_calibracao_oot.update_yaxes(range=[0, limite_calibracao_oot])
fig_calibracao_oot.show()
salvar_grafico(fig_calibracao_oot, 'final_13_calibracao_oot', PASTA_FIGURAS)
assert calibrador_final is None
print('Calibração OOT avaliada sem ajuste de Platt, Isotonic ou outro calibrador.')

modelo,amostra,brier,ece,taxa_observada,probabilidade_media,vies_absoluto,vies_relativo
CatBoost,OOT,0.0979,0.0200,14.67%,15.44%,+0.78%,+5.3%


Calibração OOT avaliada sem ajuste de Platt, Isotonic ou outro calibrador.


### 22.6 Lift, captura e monotonicidade com decis fixos

Os nove cortes internos são aprendidos exclusivamente nas probabilidades de Desenvolvimento produzidas pelo modelo final refitado. Os mesmos limites são aplicados ao OOT, sem usar o target para definir ou corrigir faixas.

In [37]:
quantis_desenvolvimento = np.quantile(
    prob_catboost_desenvolvimento,
    np.linspace(0, 1, 11),
)
assert len(np.unique(quantis_desenvolvimento)) == 11, (
    'Quantis duplicados impedem formar 10 decis sem redefinição pós-hoc.'
)
cortes_decis_desenvolvimento = np.concatenate([
    [-np.inf],
    quantis_desenvolvimento[1:-1],
    [np.inf],
])

def aplicar_decis_congelados(probabilidade: np.ndarray) -> np.ndarray:
    """Aplica cortes ascendentes e numera o maior risco como decil 1."""

    codigos_ascendentes = pd.cut(
        pd.Series(np.asarray(probabilidade, dtype=float)),
        bins=cortes_decis_desenvolvimento,
        labels=False,
        include_lowest=True,
        right=True,
    )
    if codigos_ascendentes.isna().any():
        raise ValueError('Probabilidade fora dos cortes congelados.')
    return (10 - codigos_ascendentes.astype(int)).to_numpy()

registros_cortes_decis = []
for decil in range(1, 11):
    codigo_ascendente = 10 - decil
    registros_cortes_decis.append({
        'decil_risco': decil,
        'limite_inferior_exclusivo': cortes_decis_desenvolvimento[codigo_ascendente],
        'limite_superior_inclusivo': cortes_decis_desenvolvimento[codigo_ascendente + 1],
    })
tabela_cortes_decis_desenvolvimento = pd.DataFrame(registros_cortes_decis)

quadro_lift_oot = pd.DataFrame({
    'alvo': y_oot.to_numpy(),
    'probabilidade': prob_catboost_oot,
    'decil_risco': aplicar_decis_congelados(prob_catboost_oot),
})
tabela_lift_captura_oot = (
    quadro_lift_oot.groupby('decil_risco', as_index=True)
    .agg(
        populacao=('alvo', 'size'),
        eventos=('alvo', 'sum'),
        taxa_evento=('alvo', 'mean'),
        probabilidade_media=('probabilidade', 'mean'),
    )
    .reindex(range(1, 11))
    .reset_index()
)
tabela_lift_captura_oot['populacao'] = (
    tabela_lift_captura_oot['populacao'].fillna(0).astype(int)
)
tabela_lift_captura_oot['eventos'] = (
    tabela_lift_captura_oot['eventos'].fillna(0).astype(int)
)
tabela_lift_captura_oot['participacao_populacao'] = (
    tabela_lift_captura_oot['populacao'] / len(quadro_lift_oot)
)
taxa_evento_oot = float(quadro_lift_oot['alvo'].mean())
tabela_lift_captura_oot['lift'] = (
    tabela_lift_captura_oot['taxa_evento'] / taxa_evento_oot
)
tabela_lift_captura_oot['captura_acumulada'] = (
    tabela_lift_captura_oot['eventos'].cumsum() / quadro_lift_oot['alvo'].sum()
)

taxas_validas = tabela_lift_captura_oot['taxa_evento'].dropna()
diferencas_taxa = taxas_validas.diff().dropna()
quantidade_inversoes = int((diferencas_taxa > 0).sum())
monotonicidade_oot = quantidade_inversoes == 0

display(tabela_cortes_decis_desenvolvimento.style.format({
    'limite_inferior_exclusivo': '{:.6f}',
    'limite_superior_inclusivo': '{:.6f}',
}).hide(axis='index'))
display(tabela_lift_captura_oot.style.format({
    'taxa_evento': '{:.2%}',
    'probabilidade_media': '{:.2%}',
    'participacao_populacao': '{:.2%}',
    'lift': '{:.2f}',
    'captura_acumulada': '{:.2%}',
}).hide(axis='index'))

fig_lift_oot = go.Figure(go.Bar(
    x=tabela_lift_captura_oot['decil_risco'],
    y=tabela_lift_captura_oot['lift'],
    marker_color=CORES['principal'],
))
aplicar_layout_executivo(
    fig_lift_oot,
    'Lift por decil fixo no OOT',
    subtitulo='Cortes definidos exclusivamente no Desenvolvimento; decil 1 = maior risco',
    titulo_eixo_x='Decil de risco',
    titulo_eixo_y='Lift',
    mostrar_legenda=False,
)
fig_lift_oot.update_layout(title_x=0.02, margin=dict(l=70, r=40, t=125, b=65))
fig_lift_oot.update_yaxes(rangemode='tozero')
fig_lift_oot.show()
salvar_grafico(fig_lift_oot, 'final_14_lift_oot', PASTA_FIGURAS)

fig_captura_oot = go.Figure(go.Scatter(
    x=tabela_lift_captura_oot['decil_risco'],
    y=tabela_lift_captura_oot['captura_acumulada'],
    mode='lines+markers',
    line=dict(color=CORES['principal'], width=3),
))
aplicar_layout_executivo(
    fig_captura_oot,
    'Captura acumulada de eventos no OOT',
    subtitulo='Cortes definidos exclusivamente no Desenvolvimento; decil 1 = maior risco',
    titulo_eixo_x='Decil de risco',
    titulo_eixo_y='Captura acumulada',
    mostrar_legenda=False,
)
aplicar_eixo_percentual(fig_captura_oot)
fig_captura_oot.update_layout(
    title_x=0.02,
    title_xanchor='left',
    margin=dict(l=80, r=40, t=125, b=65),
)
fig_captura_oot.show()
salvar_grafico(fig_captura_oot, 'final_15_captura_oot', PASTA_FIGURAS)

display(Markdown(
    f"**FATO OBSERVADO:** a ordenação por taxa do evento é "
    f"**{'monotônica' if monotonicidade_oot else 'não monotônica'}** nos decis fixos do Desenvolvimento. "
    f"Foram observadas **{quantidade_inversoes} inversões** entre faixas adjacentes. "
    "Nenhuma faixa foi alterada após observar o OOT."
))

decil_risco,limite_inferior_exclusivo,limite_superior_inclusivo
1,0.295535,inf
2,0.152675,0.295535
3,0.101013,0.152675
4,0.072436,0.101013
5,0.050885,0.072436
6,0.036643,0.050885
7,0.028381,0.036643
8,0.024376,0.028381
9,0.022227,0.024376
10,-inf,0.022227


decil_risco,populacao,eventos,taxa_evento,probabilidade_media,participacao_populacao,lift,captura_acumulada
1,7827,4013,51.27%,60.49%,14.74%,3.50,51.53%
2,5864,1329,22.66%,20.98%,11.04%,1.55,68.60%
3,6018,783,13.01%,12.36%,11.33%,0.89,78.66%
4,5695,542,9.52%,8.59%,10.73%,0.65,85.62%
5,5542,383,6.91%,6.10%,10.44%,0.47,90.54%
6,5287,242,4.58%,4.33%,9.96%,0.31,93.64%
7,4859,179,3.68%,3.22%,9.15%,0.25,95.94%
8,4346,113,2.60%,2.61%,8.18%,0.18,97.39%
9,3920,115,2.93%,2.33%,7.38%,0.20,98.87%
10,3740,88,2.35%,1.99%,7.04%,0.16,100.00%


**FATO OBSERVADO:** a ordenação por taxa do evento é **não monotônica** nos decis fixos do Desenvolvimento. Foram observadas **1 inversões** entre faixas adjacentes. Nenhuma faixa foi alterada após observar o OOT.

### 22.7 Estabilidade das features — diagnóstico

O PSI usa Desenvolvimento como referência do modelo final. Os bins numéricos e as categorias são definidos nessa referência e aplicados ao OOT agregado e às três safras. Nenhuma feature é removida a partir deste diagnóstico.

In [38]:
comparacoes_psi = {'OOT agregado': X_candidato_oot}
for safra, grupo in oot.groupby('safra', sort=True):
    comparacoes_psi[safra.strftime('%Y-%m')] = X_candidato_oot.loc[grupo.index]

registros_psi_oot = []
for variavel in colunas_efetivas_congeladas:
    tipo = 'numerica' if variavel in numericas_finais_congeladas else 'categorica'
    for nome_comparacao, quadro_comparacao in comparacoes_psi.items():
        if tipo == 'numerica':
            psi = calcular_psi_numerico(
                X_candidato_desenvolvimento[variavel],
                quadro_comparacao[variavel],
            )
        else:
            psi = calcular_psi_categorico(
                X_candidato_desenvolvimento[variavel],
                quadro_comparacao[variavel],
            )
        registros_psi_oot.append({
            'feature': variavel,
            'tipo': tipo,
            'comparacao': nome_comparacao,
            'psi': psi,
        })
tabela_psi_features_oot = pd.DataFrame(registros_psi_oot)
ordem_comparacoes_psi = ['OOT agregado', '2019-11', '2019-12', '2020-01']
matriz_psi_oot = (
    tabela_psi_features_oot.pivot(
        index='feature', columns='comparacao', values='psi'
    )
    .reindex(index=colunas_efetivas_congeladas, columns=ordem_comparacoes_psi)
)

fig_psi_features_oot = px.imshow(
    matriz_psi_oot,
    aspect='auto',
    color_continuous_scale=[CORES['fundo'], CORES['destaque']],
    labels={
        'x': 'Comparação',
        'y': 'Feature',
        'color': 'PSI',
    },
)
aplicar_layout_executivo(
    fig_psi_features_oot,
    'PSI das features finais no OOT',
    subtitulo='Referência: Desenvolvimento; bins e categorias não são reajustados no OOT',
    titulo_eixo_x='OOT agregado e safra',
    titulo_eixo_y='Feature',
    altura=650,
)
fig_psi_features_oot.update_layout(title_x=0.02, margin=dict(l=140, r=80, t=125, b=80))
fig_psi_features_oot.update_xaxes(
    type='category',
    tickmode='array',
    tickvals=ordem_comparacoes_psi,
    ticktext=ordem_comparacoes_psi,
)
fig_psi_features_oot.show()
salvar_grafico(fig_psi_features_oot, 'final_16_psi_features_oot', PASTA_FIGURAS)

top_drifts_oot = (
    tabela_psi_features_oot.query("comparacao == 'OOT agregado'")
    .sort_values('psi', ascending=False)
    .reset_index(drop=True)
)
display(top_drifts_oot.head(10).style.format({'psi': '{:.4f}'}).hide(axis='index'))
linha_cat_var10_psi = top_drifts_oot.query("feature == 'cat_var10'").iloc[0]
rank_cat_var10_psi = int(
    top_drifts_oot.index[top_drifts_oot['feature'].eq('cat_var10')][0] + 1
)
display(Markdown(
    f"**FATO OBSERVADO:** `cat_var10` ocupa a posição **{rank_cat_var10_psi}** entre os "
    f"PSIs agregados, com PSI **{linha_cat_var10_psi['psi']:.4f}**. O ranking é diagnóstico; "
    "não foi aplicado threshold de exclusão."
))

feature,tipo,comparacao,psi
cat_var10,categorica,OOT agregado,0.1483
var3,numerica,OOT agregado,0.0422
var7,numerica,OOT agregado,0.0409
var1,numerica,OOT agregado,0.0206
var12_estado,categorica,OOT agregado,0.0196
cat_var15,categorica,OOT agregado,0.0194
var14,numerica,OOT agregado,0.0165
cat_var2,categorica,OOT agregado,0.0029
var4,numerica,OOT agregado,0.0028
var9,numerica,OOT agregado,0.0027


**FATO OBSERVADO:** `cat_var10` ocupa a posição **1** entre os PSIs agregados, com PSI **0.1483**. O ranking é diagnóstico; não foi aplicado threshold de exclusão.

### 22.8 SHAP OOT — estabilidade explicativa

O SHAP é calculado em amostras aleatórias determinísticas de até 3.000 registros de Desenvolvimento e OOT. A comparação é somente diagnóstica e não pode remover features nem alterar o modelo.

In [39]:
amostra_shap_desenvolvimento_final = X_candidato_desenvolvimento.sample(
    n=min(3000, len(X_candidato_desenvolvimento)),
    random_state=42,
)
amostra_shap_oot_final = X_candidato_oot.sample(
    n=min(3000, len(X_candidato_oot)),
    random_state=42,
)

valores_shap_desenvolvimento_final = modelo_catboost_final.get_feature_importance(
    Pool(
        amostra_shap_desenvolvimento_final,
        cat_features=categoricas_candidato_final,
    ),
    type='ShapValues',
)[:, :-1]
valores_shap_oot_final = modelo_catboost_final.get_feature_importance(
    Pool(
        amostra_shap_oot_final,
        cat_features=categoricas_candidato_final,
    ),
    type='ShapValues',
)[:, :-1]

shap_desenvolvimento = pd.DataFrame({
    'feature': amostra_shap_desenvolvimento_final.columns,
    'shap_abs_medio_desenvolvimento': np.abs(
        valores_shap_desenvolvimento_final
    ).mean(axis=0),
})
shap_oot = pd.DataFrame({
    'feature': amostra_shap_oot_final.columns,
    'shap_abs_medio_oot': np.abs(valores_shap_oot_final).mean(axis=0),
})
tabela_shap_desenvolvimento_oot = shap_desenvolvimento.merge(
    shap_oot, on='feature', validate='one_to_one'
)
tabela_shap_desenvolvimento_oot['delta_shap_abs_medio'] = (
    tabela_shap_desenvolvimento_oot['shap_abs_medio_oot']
    - tabela_shap_desenvolvimento_oot['shap_abs_medio_desenvolvimento']
)
tabela_shap_desenvolvimento_oot['participacao_desenvolvimento'] = (
    tabela_shap_desenvolvimento_oot['shap_abs_medio_desenvolvimento']
    / tabela_shap_desenvolvimento_oot['shap_abs_medio_desenvolvimento'].sum()
)
tabela_shap_desenvolvimento_oot['participacao_oot'] = (
    tabela_shap_desenvolvimento_oot['shap_abs_medio_oot']
    / tabela_shap_desenvolvimento_oot['shap_abs_medio_oot'].sum()
)
tabela_shap_desenvolvimento_oot['rank_desenvolvimento'] = (
    tabela_shap_desenvolvimento_oot['shap_abs_medio_desenvolvimento']
    .rank(method='min', ascending=False)
    .astype(int)
)
tabela_shap_desenvolvimento_oot['rank_oot'] = (
    tabela_shap_desenvolvimento_oot['shap_abs_medio_oot']
    .rank(method='min', ascending=False)
    .astype(int)
)
tabela_shap_desenvolvimento_oot['mudanca_rank'] = (
    tabela_shap_desenvolvimento_oot['rank_oot']
    - tabela_shap_desenvolvimento_oot['rank_desenvolvimento']
)
tabela_shap_desenvolvimento_oot = tabela_shap_desenvolvimento_oot.sort_values(
    'rank_oot'
).reset_index(drop=True)
display(tabela_shap_desenvolvimento_oot.style.format({
    'shap_abs_medio_desenvolvimento': '{:.4f}',
    'shap_abs_medio_oot': '{:.4f}',
    'delta_shap_abs_medio': '{:+.4f}',
    'participacao_desenvolvimento': '{:.1%}',
    'participacao_oot': '{:.1%}',
}).hide(axis='index'))

tabela_shap_plot = tabela_shap_desenvolvimento_oot.copy()
tabela_shap_plot['max_importancia'] = tabela_shap_plot[[
    'shap_abs_medio_desenvolvimento', 'shap_abs_medio_oot'
]].max(axis=1)
features_shap_plot = (
    tabela_shap_plot.nlargest(10, 'max_importancia')
    .sort_values('max_importancia')['feature']
    .tolist()
)
dados_shap_plot = (
    tabela_shap_plot[tabela_shap_plot['feature'].isin(features_shap_plot)]
    .melt(
        id_vars='feature',
        value_vars=[
            'shap_abs_medio_desenvolvimento',
            'shap_abs_medio_oot',
        ],
        var_name='amostra',
        value_name='shap_abs_medio',
    )
)
dados_shap_plot['amostra'] = dados_shap_plot['amostra'].map({
    'shap_abs_medio_desenvolvimento': 'Desenvolvimento',
    'shap_abs_medio_oot': 'OOT',
})
fig_shap_oot = px.bar(
    dados_shap_plot,
    x='shap_abs_medio',
    y='feature',
    color='amostra',
    orientation='h',
    barmode='group',
    category_orders={'feature': features_shap_plot},
    color_discrete_map={
        'Desenvolvimento': CORES['secundaria'],
        'OOT': CORES['principal'],
    },
)
aplicar_layout_executivo(
    fig_shap_oot,
    'Importância SHAP: Desenvolvimento versus OOT',
    subtitulo='Amostras aleatórias determinísticas de até 3.000 registros por população',
    titulo_eixo_x='Média de |SHAP|',
    titulo_eixo_y='Feature',
)
fig_shap_oot.update_layout(
    title_x=0.02,
    margin=dict(l=150, r=40, t=150, b=65),
    legend=dict(orientation='h', yanchor='bottom', y=1.0, xanchor='left', x=0.02),
)
fig_shap_oot.show()
salvar_grafico(fig_shap_oot, 'final_17_shap_oot', PASTA_FIGURAS)

top5_shap_dev = set(
    tabela_shap_desenvolvimento_oot.nsmallest(5, 'rank_desenvolvimento')['feature']
)
top5_shap_oot = set(
    tabela_shap_desenvolvimento_oot.nsmallest(5, 'rank_oot')['feature']
)
sobreposicao_top5 = len(top5_shap_dev & top5_shap_oot)
linha_cat10_shap = tabela_shap_desenvolvimento_oot.query(
    "feature == 'cat_var10'"
).iloc[0]
display(Markdown(
    f"**FATO OBSERVADO:** **{sobreposicao_top5} de 5** features permanecem em comum no top 5 de "
    f"Desenvolvimento e OOT. `cat_var10` muda do rank "
    f"**{int(linha_cat10_shap['rank_desenvolvimento'])}** para "
    f"**{int(linha_cat10_shap['rank_oot'])}**, e sua participação no |SHAP| total muda de "
    f"**{linha_cat10_shap['participacao_desenvolvimento']:.1%}** para "
    f"**{linha_cat10_shap['participacao_oot']:.1%}**. Nenhuma feature é removida a partir deste resultado."
))

feature,shap_abs_medio_desenvolvimento,shap_abs_medio_oot,delta_shap_abs_medio,participacao_desenvolvimento,participacao_oot,rank_desenvolvimento,rank_oot,mudanca_rank
var1,0.5067,0.5709,+0.0642,22.2%,22.7%,1,1,0
var3,0.4505,0.5190,+0.0685,19.7%,20.6%,2,2,0
cat_var10,0.2362,0.3169,+0.0807,10.4%,12.6%,3,3,0
cat_var2,0.2132,0.2114,-0.0018,9.3%,8.4%,4,4,0
var5,0.1906,0.1809,-0.0097,8.4%,7.2%,5,5,0
var14,0.1652,0.1741,+0.0090,7.2%,6.9%,6,6,0
var7,0.1111,0.1351,+0.0240,4.9%,5.4%,7,7,0
var9,0.0991,0.1001,+0.0009,4.3%,4.0%,8,8,0
var11,0.0870,0.0863,-0.0007,3.8%,3.4%,9,9,0
var12_estado,0.0789,0.0735,-0.0054,3.5%,2.9%,10,10,0


**FATO OBSERVADO:** **5 de 5** features permanecem em comum no top 5 de Desenvolvimento e OOT. `cat_var10` muda do rank **3** para **3**, e sua participação no |SHAP| total muda de **10.4%** para **12.6%**. Nenhuma feature é removida a partir deste resultado.

### 22.9 Artefatos e integridade

As saídas abaixo são agregadas. Probabilidades individuais, dados raw, modelo serializado e qualquer transformação de score não são persistidos nesta fase.

In [40]:
import hashlib

protocolo_refit_oot = pd.DataFrame([{
    'checkpoint_pre_oot': '0c5c6a7',
    'best_iteration_desenvolvimento': best_iteration_congelado,
    'tree_count_congelado': quantidade_arvores_congelada,
    'registros_desenvolvimento_fit': len(desenvolvimento),
    'registros_oot_fit': 0,
    'early_stopping_refit': False,
    'calibracao': 'nenhuma',
    'features_originais': len(features_originais_congeladas),
    'features_efetivas': len(colunas_efetivas_congeladas),
}])

tabelas_oot_finais = {
    'final_oot_protocolo_refit.csv': protocolo_refit_oot,
    'final_oot_comparacao_metricas.csv': tabela_comparacao_oot,
    'final_oot_calibracao_agregada.csv': tabela_calibracao_agregada_oot,
    'final_oot_intervalos_bootstrap.csv': tabela_ic_catboost_oot,
    'final_oot_performance_por_safra.csv': tabela_performance_oot_por_safra,
    'final_oot_performance_temporal.csv': tabela_performance_temporal,
    'final_oot_cortes_decis_desenvolvimento.csv': tabela_cortes_decis_desenvolvimento,
    'final_oot_lift_captura.csv': tabela_lift_captura_oot,
    'final_oot_psi_features.csv': tabela_psi_features_oot,
    'final_oot_shap_comparacao.csv': tabela_shap_desenvolvimento_oot,
}
for nome, tabela in tabelas_oot_finais.items():
    tabela.to_csv(PASTA_TABELAS / nome, index=False, encoding='utf-8-sig')

sha256_esperado_raw = '7ae6cca5ca1e488920a465c1f6fe93850c972f15a996eb1bc8f3a35783435e08'
sha256_atual_raw = hashlib.sha256(caminho_base.read_bytes()).hexdigest()
assert sha256_atual_raw == sha256_esperado_raw
assert base_original.equals(base.drop(columns=['safra', 'amostra']))
assert len(prob_catboost_oot) == len(oot)
assert len(prob_logistica_oot) == len(oot)
assert indices_fit_catboost_final.isdisjoint(indices_oot)
assert indices_fit_logistica_final.isdisjoint(indices_oot)
assert int(modelo_catboost_final.tree_count_) == quantidade_arvores_congelada
assert calibrador_final is None
assert 'score' not in base.columns
assert callable(calcular_behavior_score)

print(f'{len(tabelas_oot_finais)} tabelas OOT agregadas exportadas.')
print(f'SHA-256 raw validado: {sha256_atual_raw}')
print('Zero linhas OOT no fit; nenhum score 0–1000 calculado sem a convenção de escala.')

10 tabelas OOT agregadas exportadas.
SHA-256 raw validado: 7ae6cca5ca1e488920a465c1f6fe93850c972f15a996eb1bc8f3a35783435e08
Zero linhas OOT no fit; nenhum score 0–1000 calculado sem a convenção de escala.


## 23. Behavior Score 0–1000 — fórmula definida, parâmetros pendentes

**Comentário Técnico:** a transformação reutilizável está implementada em `src/behavior_score/scoring.py`, na função `calcular_behavior_score`. Nenhuma probabilidade de Desenvolvimento ou OOT é convertida nesta etapa, porque a convenção da escala ainda depende de decisão humana.

Para uma probabilidade do evento adverso `p`, define-se `odds(p) = (1 - p) / p`, isto é, a razão **não-evento:evento**. A escala segue:

- `fator = PDO / ln(2)`;
- `score_bruto = Base Score + fator × ln(odds(p) / Base Odds)`;
- `score = clip(score_bruto, 0, 1000)`, quando o clipping final estiver habilitado.

Os parâmetros têm papéis distintos:

- **Base Score:** ponto da escala associado à Base Odds; desloca a escala para cima ou para baixo.
- **PDO (Points to Double the Odds):** quantidade de pontos adicionada quando as odds de não-evento:evento dobram; controla a dispersão da escala.
- **Base Odds:** razão de referência não-evento:evento que recebe exatamente o Base Score; ancora a interpretação probabilística.

Como `odds(p)` diminui estritamente quando `p` aumenta, o logaritmo e o fator positivo preservam essa direção: **maior probabilidade do evento implica menor score**. A transformação bruta preserva a ordenação; o clipping em 0 e 1000 pode criar empates nas caudas, mas não cria inversões. Probabilidades iguais a 0 ou 1 são protegidas com epsilon de máquina antes do logaritmo.

**Decisão humana pendente:** escolher **A. Base Score**, **B. PDO** e **C. Base Odds**. Esses valores são convenções de escala, não hiperparâmetros preditivos, e não serão definidos ou otimizados com o OOT. Até essa escolha, não são calculados scores individuais, faixas de risco, limites ou políticas de negócio.

## 24. Conclusão

**Classificação técnica: B — boa capacidade de ordenação fora do tempo, com variabilidade temporal material.**

O CatBoost final refitado alcança ROC-AUC **0,8227** (IC95% bootstrap **[0,8171; 0,8278]**), Gini **0,6455**, KS **0,5027**, PR-AUC/AP **0,4987** e Brier **0,0979** no OOT. O primeiro decil fixo apresenta lift **3,50** e captura **51,5%** dos eventos; os dois primeiros capturam **68,6%**. O CatBoost permanece acima da Regressão Logística em todas as métricas de discriminação e com Brier/ECE menores, portanto não há fundamento técnico para troca do champion congelado pelo benchmark.

A diferença entre a referência de desenvolvimento e a avaliação OOT do modelo final refitado é material, mas deve ser lida entre ajustes distintos: o ROC-AUC de **0,8421** vem do candidato treinado no Treino e avaliado na Validação, enquanto **0,8227** vem do modelo final refitado em Treino + Validação e avaliado no OOT, uma diferença de **0,0194**. Entre esses referenciais, as diferenças também são **-0,0275** em KS, **-0,0857** em PR-AUC e **+0,0219** em Brier. Dezembro de 2019 é a safra mais fraca, com ROC-AUC **0,7758** e ECE **0,0374**, enquanto janeiro de 2020 apresenta ROC-AUC **0,8588**. Os ECE de referência são **0,0093** na Validação e **0,0200** no OOT. Há uma inversão adjacente entre os decis fixos 8 e 9; nenhum corte foi ajustado após observá-la.

O principal deslocamento agregado é `cat_var10`, com PSI **0,1483** e crescimento mensal, mas sua importância SHAP permanece em terceiro lugar, atrás de `var1` e `var3`. O top 5 SHAP coincide integralmente entre Desenvolvimento e OOT, sem evidência de troca estrutural abrupta nos principais direcionadores. Nenhum resultado OOT alterou features, tratamentos, hiperparâmetros, calibração ou quantidade de árvores.

A conclusão é limitada a uma única janela OOT de três meses, a variáveis e target anonimizados e à ausência de avaliação econômica ou operacional. A decisão pré-OOT de remover `cat_var13` permanece uma exceção humana à regra quantitativa declarada e não é reavaliada nesta fase. A avaliação OOT foi aprovada por auditoria independente. A fórmula do score 0–1000 está implementada, mas Base Score, PDO e Base Odds permanecem pendentes; por isso, não foram calculados scores individuais, faixas finais de risco nem política de uso.